In [3]:
"""
place_lights.py
---------------
Places lighting fixtures in AutoCAD using pyautocad.
Edit the BLOCK_NAMES and PLACEMENTS sections below, then run the script
while your AutoCAD drawing is open.

Requirements:
    pip install pyautocad
"""

from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# 1.  MAP each fixture to its AutoCAD block name
#     Change the values to match your actual block names.
# ─────────────────────────────────────────────
BLOCK_NAMES = {
    "panneau_led":      "PANNEAU_LED_60x60",   # Square LED panel 40W VT-6170
    "spot_downlight":   "SPOT_CORELINE_DN140B", # CoreLine Downlight 18W
    "hublot_etanche":   "HUBLOT_ETANCHE_11W",  # Hublot LEGRAND 11W
    "applique_etanche": "APPLIQUE_ETANCHE_11W", # Applique LEGRAND 500276
}

# ─────────────────────────────────────────────
# 2.  PLACEMENTS — add as many rows as you need.
#
#     Format:  ("fixture_key", x, y, z, scale, rotation_degrees)
#
#     fixture_key  : one of the keys in BLOCK_NAMES above
#     x, y, z      : insertion point in drawing units
#     scale        : uniform scale factor  (1.0 = no change)
#     rotation_deg : rotation in degrees   (0 = no rotation)
# ─────────────────────────────────────────────
PLACEMENTS = [
    # key                  x       y     z    scale  rot°
    ("panneau_led",        1000,   2000, 0,   1.0,   0),
    ("panneau_led",        1600,   2000, 0,   1.0,   0),
    ("spot_downlight",     1000,   1400, 0,   1.0,   0),
    ("spot_downlight",     1600,   1400, 0,   1.0,   0),
    ("hublot_etanche",     2200,   2000, 0,   1.0,   0),
    ("applique_etanche",   2200,   1400, 0,   1.0,   0),
]

# ─────────────────────────────────────────────
# 3.  LAYER settings — one layer per fixture type.
#     Set to None to use the current active layer.
# ─────────────────────────────────────────────
LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

# ═════════════════════════════════════════════
# — Do not edit below unless you know what
#   you are doing —
# ═════════════════════════════════════════════

import math

def ensure_layer(doc, layer_name: str):
    """Create the layer if it doesn't exist yet."""
    try:
        doc.Layers.Item(layer_name)
    except Exception:
        doc.Layers.Add(layer_name)


def place_fixtures():
    acad = Autocad(create_if_not_exists=False)
    doc  = acad.doc
    ms   = acad.model   # ModelSpace

    print(f"Connected to: {doc.Name}")
    print(f"Placing {len(PLACEMENTS)} fixture(s)...\n")

    placed   = 0
    skipped  = 0

    for idx, (key, x, y, z, scale, rot_deg) in enumerate(PLACEMENTS, start=1):
        block_name = BLOCK_NAMES.get(key)
        if not block_name:
            print(f"  [{idx}] SKIP — unknown fixture key: '{key}'")
            skipped += 1
            continue

        # Verify the block exists in the drawing
        try:
            doc.Blocks.Item(block_name)
        except Exception:
            print(f"  [{idx}] SKIP — block '{block_name}' not found in drawing.")
            skipped += 1
            continue

        # Switch layer if configured
        layer = LAYER_MAP.get(key)
        if layer:
            ensure_layer(doc, layer)
            doc.ActiveLayer = doc.Layers.Item(layer)

        # Insert the block reference
        insertion_point = APoint(x, y, z)
        rot_rad = math.radians(rot_deg)

        ref = ms.InsertBlock(insertion_point, block_name, scale, scale, scale, rot_rad)

        print(f"  [{idx}] OK — '{key}' ({block_name}) at ({x}, {y}, {z})"
              f"  scale={scale}  rot={rot_deg}°  layer={layer or 'active'}")
        placed += 1

    # Restore default layer
    doc.ActiveLayer = doc.Layers.Item("0")

    doc.Application.ZoomExtents()
    print(f"\nDone. {placed} placed, {skipped} skipped.")


if __name__ == "__main__":
    place_fixtures()

Connected to: Drawing1.dwg
Placing 6 fixture(s)...

  [1] OK — 'panneau_led' (PANNEAU_LED_60x60) at (1000, 2000, 0)  scale=1.0  rot=0°  layer=ECLAIRAGE-PANNEAU
  [2] OK — 'panneau_led' (PANNEAU_LED_60x60) at (1600, 2000, 0)  scale=1.0  rot=0°  layer=ECLAIRAGE-PANNEAU
  [3] SKIP — block 'SPOT_CORELINE_DN140B' not found in drawing.
  [4] SKIP — block 'SPOT_CORELINE_DN140B' not found in drawing.
  [5] SKIP — block 'HUBLOT_ETANCHE_11W' not found in drawing.
  [6] SKIP — block 'APPLIQUE_ETANCHE_11W' not found in drawing.

Done. 2 placed, 4 skipped.


In [18]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline with 4 vertices) in the drawing
2. For each rectangle, places a chosen light fixture in an evenly spaced grid
3. Fixes the "disappearing blocks" issue by using doc.Regen + doc.Save

Requirements:
    pip install pyautocad pywin32

Usage:
    - Open your DWG in AutoCAD
    - Draw closed rectangles (RECTANG command) for each room/zone
    - Run:  python place_lights_auto.py
"""

import math
import sys
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION — edit these
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

# Layer name of rectangles to scan (None = scan all layers)
RECTANGLE_LAYER = None

# Block insertion scale
BLOCK_SCALE = 1.0

# Margin from rectangle edges (in drawing units)
MARGIN = 50.0

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name):
    try:
        doc.Layers.Item(name)
    except Exception:
        doc.Layers.Add(name)


def get_rectangle_bounds(pline):
    """
    Extract (x_min, y_min, x_max, y_max) from a closed 4-vertex LWPolyline.
    Returns None if the entity is not a valid rectangle.
    """
    try:
        if pline.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        coords = list(pline.Coordinates)   # flat list [x0,y0, x1,y1, ...]
        # LWPolyline stores 2D coords; 2dPolyline stores 3D
        step = 2 if pline.EntityName == "AcDbPolyline" else 3
        pts = [(coords[i], coords[i + 1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        return min(xs), min(ys), max(xs), max(ys)
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return n points arranged in the most square grid possible,
    evenly distributed inside the rectangle with margin applied.
    """
    x0 = x_min + MARGIN
    y0 = y_min + MARGIN
    x1 = x_max - MARGIN
    y1 = y_max - MARGIN

    if x1 <= x0 or y1 <= y0:
        # Margin too large — fall back to centre point repeated
        cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
        return [(cx, cy)] * n

    width  = x1 - x0
    height = y1 - y0

    # Find cols × rows closest to the rectangle aspect ratio
    best = None
    best_waste = float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        if rows == 0:
            continue
        # How well does cols/rows match width/height ratio?
        ratio_diff = abs((cols / rows) - (width / height))
        waste = cols * rows - n
        score = ratio_diff + waste * 0.1
        if score < best_waste:
            best_waste = score
            best = (cols, rows)

    cols, rows = best

    # Spacing
    sx = width  / cols  if cols > 1 else width
    sy = height / rows  if rows > 1 else height
    ox = x0 + sx / 2   if cols > 1 else x0 + width  / 2
    oy = y0 + sy / 2   if rows > 1 else y0 + height / 2

    pts = []
    for r in range(rows):
        for c in range(cols):
            if len(pts) >= n:
                break
            pts.append((ox + c * sx, oy + r * sy))

    return pts


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # ── Connect to AutoCAD ──────────────────
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # ── Scan for rectangles ──────────────────
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed rectangles (4-vertex polylines) found in the drawing.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"  size: {x1-x0:.1f} × {y1-y0:.1f}")

    # ── Ask which rectangles to use ──────────
    sel = input(
        "\nWhich rectangles to fill? "
        "(e.g. 1,2,3  or  'all'): "
    ).strip().lower()

    if sel == "all":
        chosen = list(range(len(rectangles)))
    else:
        chosen = []
        for s in sel.split(","):
            try:
                idx = int(s.strip()) - 1
                if 0 <= idx < len(rectangles):
                    chosen.append(idx)
            except ValueError:
                pass

    if not chosen:
        print("No valid selection. Exiting.")
        sys.exit(0)

    # ── Pick fixture ─────────────────────────
    fixture_name, block_name = pick_fixture()

    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found in the drawing.")
        print("Make sure the block is defined or insert it once manually first.")
        sys.exit(1)

    # ── Ask for number of lights ─────────────
    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0:
                break
        except ValueError:
            pass
        print("  Please enter a positive integer.")

    # ── Prepare layer ────────────────────────
    layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, layer)

    # ── Place lights ─────────────────────────
    total = 0
    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts = grid_points(x0, y0, x1, y1, n_lights)

        doc.ActiveLayer = doc.Layers.Item(layer)

        refs = []  # keep references alive to prevent COM garbage collection
        for (px, py) in pts:
            ref = ms.InsertBlock(
                APoint(px, py, 0),
                block_name,
                BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE,
                0.0          # rotation
            )
            refs.append(ref)   # hold reference in memory
            total += 1

        print(f"  Rectangle [{idx+1}]: placed {len(pts)} × '{fixture_name}'")

    # ── Commit everything to AutoCAD ─────────
    # Keeping refs[] alive above prevents COM GC from wiping blocks.
    # Regen + Save makes them permanent.
    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)              # full regen — makes blocks stick
    doc.Save()                   # write to disk

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing2.dwg

Found 1 rectangle(s):
  [1] (12.9, 11.0) → (17.8, 15.4)  size: 4.8 × 4.3

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W
  Rectangle [1]: placed 1 × 'spot_downlight'

✓ Done. 1 light(s) placed and saved.


In [13]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline) in the drawing
2. For each rectangle, places a chosen light fixture in an evenly spaced grid
3. Uses GetBoundingBox (reliable) instead of .Coordinates (buggy COM variant)

Requirements:
    pip install pyautocad pywin32

Usage:
    - Open your DWG in AutoCAD
    - Draw closed rectangles (RECTANG command) for each room/zone
    - Run:  python place_lights_auto.py
"""

import math
import sys
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION — edit these
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

# Only scan this layer for rectangles (None = all layers)
RECTANGLE_LAYER = None

# Block insertion scale
BLOCK_SCALE = 1.0

# Margin from rectangle edges (in drawing units)
MARGIN = 50.0

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name):
    try:
        doc.Layers.Item(name)
    except Exception:
        doc.Layers.Add(name)


def get_rectangle_bounds(entity):
    """
    Returns (x_min, y_min, x_max, y_max) using GetBoundingBox.
    This avoids the broken COM variant returned by .Coordinates.
    Works on any closed polyline (RECTANG creates AcDbPolyline).
    """
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None

        min_pt, max_pt = entity.GetBoundingBox()

        x0 = float(min_pt[0])
        y0 = float(min_pt[1])
        x1 = float(max_pt[0])
        y1 = float(max_pt[1])

        # Reject degenerate shapes (lines, points)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None

        return x0, y0, x1, y1

    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return exactly n points in an evenly spaced grid inside the rectangle.
    Picks cols x rows whose aspect ratio best matches the room shape.
    """
    x0 = x_min + MARGIN
    y0 = y_min + MARGIN
    x1 = x_max - MARGIN
    y1 = y_max - MARGIN

    # If margin is too large just centre all lights
    if x1 <= x0 or y1 <= y0:
        cx = (x_min + x_max) / 2
        cy = (y_min + y_max) / 2
        return [(cx, cy)] * n

    width  = x1 - x0
    height = y1 - y0
    aspect = width / height  # > 1 means wider than tall

    # Find the cols x rows layout whose aspect ratio is closest to the room
    best_cols, best_rows = 1, n
    best_score = float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        score = abs((cols / rows) - aspect)
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows

    # Cell size
    cell_w = width  / cols
    cell_h = height / rows

    # Place lights at cell centres
    pts = []
    for r in range(rows):
        for c in range(cols):
            if len(pts) >= n:
                break
            px = x0 + cell_w * c + cell_w / 2
            py = y0 + cell_h * r + cell_h / 2
            pts.append((px, py))

    return pts[:n]  # exact n, no overflow


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # ── Connect ─────────────────────────────
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # ── Scan for rectangles ──────────────────
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed rectangles found. Use the RECTANG command to draw room outlines.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"   {x1-x0:.1f} × {y1-y0:.1f} units")

    # ── Choose rectangles ────────────────────
    sel = input("\nWhich rectangles to fill? (e.g. 1,2  or  'all'): ").strip().lower()
    if sel == "all":
        chosen = list(range(len(rectangles)))
    else:
        chosen = []
        for s in sel.split(","):
            try:
                idx = int(s.strip()) - 1
                if 0 <= idx < len(rectangles):
                    chosen.append(idx)
            except ValueError:
                pass

    if not chosen:
        print("No valid selection. Exiting.")
        sys.exit(0)

    # ── Choose fixture ───────────────────────
    fixture_name, block_name = pick_fixture()

    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found in the drawing.")
        print("Tip: Insert the block manually once so AutoCAD registers it.")
        sys.exit(1)

    # ── Number of lights ─────────────────────
    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0:
                break
        except ValueError:
            pass
        print("  Please enter a positive integer.")

    # ── Prepare layer ────────────────────────
    layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, layer)

    # ── Place lights ─────────────────────────
    total = 0
    refs  = []   # keep COM references alive until doc.Save()

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts = grid_points(x0, y0, x1, y1, n_lights)

        doc.ActiveLayer = doc.Layers.Item(layer)

        for (px, py) in pts:
            ref = ms.InsertBlock(
                APoint(px, py, 0),
                block_name,
                BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE,
                0.0
            )
            refs.append(ref)   # prevent COM garbage collection
            total += 1

        print(f"  Rectangle [{idx+1}]: placed {len(pts)} × '{fixture_name}'"
              f"  grid inside ({x0:.1f},{y0:.1f})→({x1:.1f},{y1:.1f})")

    # ── Commit ───────────────────────────────
    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg
No closed rectangles found. Use the RECTANG command to draw room outlines.


SystemExit: 0

In [27]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline with 4 vertices) in the drawing
2. For each rectangle, places a chosen light fixture in an evenly spaced grid
3. Grid is computed from the rectangle's own coordinates — no fixed margin needed

Requirements:
    pip install pyautocad pywin32

Usage:
    - Open your DWG in AutoCAD
    - Draw closed rectangles (RECTANG command) for each room/zone
    - Run:  python place_lights_auto.py
"""

import math
import sys
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION — edit these
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

# Layer name of rectangles to scan (None = scan all layers)
RECTANGLE_LAYER = None

# Block insertion scale
BLOCK_SCALE = 1.0

# Margin as a fraction of rectangle dimension (0.1 = 10%)
# Applied per-axis so it always fits regardless of drawing units
MARGIN_RATIO = 0.0005

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name):
    try:
        doc.Layers.Item(name)
    except Exception:
        doc.Layers.Add(name)


def get_rectangle_bounds(pline):
    """
    Extract (x_min, y_min, x_max, y_max) from a closed 4-vertex LWPolyline.
    Returns None if the entity is not a valid rectangle.
    """
    try:
        if pline.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        coords = list(pline.Coordinates)
        step = 2 if pline.EntityName == "AcDbPolyline" else 3
        pts = [(coords[i], coords[i + 1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        return min(xs), min(ys), max(xs), max(ys)
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return exactly n points in a well-distributed grid inside the rectangle.

    Strategy
    --------
    1. Apply a proportional margin (MARGIN_RATIO × dimension) on each side
       so the grid always stays inside regardless of drawing units.
    2. Find the cols × rows combination whose aspect ratio best matches the
       available area, minimising empty cells.
    3. Divide the usable area into cols × rows equal cells and place one
       light at the centre of each cell (up to n lights).

    This guarantees:
    - Lights are always INSIDE the rectangle.
    - Spacing is perfectly uniform in both X and Y.
    - The grid shape matches the room shape (wide room → more columns).
    """
    width  = x_max - x_min
    height = y_max - y_min

    # Proportional margin — scales with the rectangle size
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO

    # Usable area after margin
    ux0 = x_min + mx
    uy0 = y_min + my
    ux1 = x_max - mx
    uy1 = y_max - my

    uw = ux1 - ux0   # usable width
    uh = uy1 - uy0   # usable height

    # ── Find best cols / rows ────────────────────────────────────────────
    # We want cols/rows ≈ uw/uh (match room aspect ratio) with minimum waste.
    best_cols, best_rows = 1, n
    best_score = float("inf")

    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        # Penalise deviation from room aspect ratio
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5   # waste penalty
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows

    # ── Cell size ────────────────────────────────────────────────────────
    # Divide usable area into cols × rows equal cells.
    cell_w = uw / cols
    cell_h = uh / rows

    # ── Generate cell centres ────────────────────────────────────────────
    pts = []
    for r in range(rows):
        for c in range(cols):
            if len(pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            pts.append((cx, cy))
        if len(pts) >= n:
            break

    return pts


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # ── Connect to AutoCAD ──────────────────
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # ── Scan for rectangles ──────────────────
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed rectangles (4-vertex polylines) found in the drawing.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.3f}, {y0:.3f}) → ({x1:.3f}, {y1:.3f})"
              f"  size: {x1-x0:.3f} × {y1-y0:.3f}")

    # ── Ask which rectangles to use ──────────
    sel = input(
        "\nWhich rectangles to fill? "
        "(e.g. 1,2,3  or  'all'): "
    ).strip().lower()

    if sel == "all":
        chosen = list(range(len(rectangles)))
    else:
        chosen = []
        for s in sel.split(","):
            try:
                idx = int(s.strip()) - 1
                if 0 <= idx < len(rectangles):
                    chosen.append(idx)
            except ValueError:
                pass

    if not chosen:
        print("No valid selection. Exiting.")
        sys.exit(0)

    # ── Pick fixture ─────────────────────────
    fixture_name, block_name = pick_fixture()

    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found in the drawing.")
        print("Make sure the block is defined or insert it once manually first.")
        sys.exit(1)

    # ── Ask for number of lights ─────────────
    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0:
                break
        except ValueError:
            pass
        print("  Please enter a positive integer.")

    # ── Prepare layer ────────────────────────
    layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, layer)

    # ── Place lights ─────────────────────────
    total = 0
    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts = grid_points(x0, y0, x1, y1, n_lights)

        # Debug: confirm all points are inside the rectangle
        for px, py in pts:
            assert x0 < px < x1, f"X out of bounds: {px} not in ({x0}, {x1})"
            assert y0 < py < y1, f"Y out of bounds: {py} not in ({y0}, {y1})"
            print(f"    → light at ({px:.4f}, {py:.4f})")

        doc.ActiveLayer = doc.Layers.Item(layer)

        refs = []  # keep COM references alive to prevent garbage collection
        for (px, py) in pts:
            ref = ms.InsertBlock(
                APoint(px, py, 0),
                block_name,
                BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE,
                0.0   # rotation in radians
            )
            refs.append(ref)
            total += 1

        print(f"  Rectangle [{idx+1}]: placed {len(pts)} × '{fixture_name}'"
              f"  (grid: see points above)")

    # ── Commit everything to AutoCAD ─────────
    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing2.dwg

Found 3 rectangle(s):
  [1] (15.665, 26.595) → (162.235, 64.853)  size: 146.570 × 38.258
  [2] (1.601, 85.807) → (148.171, 124.065)  size: 146.570 × 38.258
  [3] (25.728, -19.818) → (172.298, 18.440)  size: 146.570 × 38.258

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W
    → light at (40.4435, -10.2442)
    → light at (69.7281, -10.2442)
    → light at (99.0127, -10.2442)
    → light at (128.2973, -10.2442)
    → light at (157.5819, -10.2442)
    → light at (40.4435, 8.8657)
    → light at (69.7281, 8.8657)
    → light at (99.0127, 8.8657)
    → light at (128.2973, 8.8657)
    → light at (157.5819, 8.8657)
  Rectangle [3]: placed 10 × 'spot_downlight'  (grid: see points above)

✓ Done. 10 light(s) placed and saved.


In [30]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline with 4 vertices) in the drawing
2. For each rectangle, places a chosen light fixture in an evenly spaced grid
3. Draws a spline wire connecting all lights in reading order (row by row)
4. Adds a text label (circuit name) below each light
5. Grid is computed from the rectangle's own coordinates — no fixed margin needed

Requirements:
    pip install pyautocad pywin32

Usage:
    - Open your DWG in AutoCAD
    - Draw closed rectangles (RECTANG command) for each room/zone
    - Run:  python place_lights_auto.py
"""

import math
import sys
import pythoncom
import win32com.client

from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION — edit these
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"       # layer for spline wire
LABEL_LAYER = "ECLAIRAGE-LABEL"      # layer for text labels

# Layer name of rectangles to scan (None = scan all layers)
RECTANGLE_LAYER = None

# Block insertion scale
BLOCK_SCALE = 1.0

# Margin as a fraction of rectangle dimension (0.1 = 10%)
MARGIN_RATIO = 0.10

# Label text height — fraction of rectangle's smaller dimension
LABEL_HEIGHT_RATIO = 0.06

# Label offset below block insertion point (fraction of rectangle height)
LABEL_OFFSET_RATIO = 0.05


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(pline):
    try:
        if pline.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        coords = list(pline.Coordinates)
        step = 2 if pline.EntityName == "AcDbPolyline" else 3
        pts = [(coords[i], coords[i + 1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        return min(xs), min(ys), max(xs), max(ys)
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts, cols, rows) where pts is exactly n (x,y) points
    arranged in a grid that matches the rectangle's aspect ratio.

    Wire order: row by row, left→right, bottom→top so the spline
    snakes naturally through the room like in the reference image.
    """
    width  = x_max - x_min
    height = y_max - y_min

    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO

    ux0 = x_min + mx
    uy0 = y_min + my
    ux1 = x_max - mx
    uy1 = y_max - my

    uw = ux1 - ux0
    uh = uy1 - uy0

    # Find best cols × rows
    best_cols, best_rows = 1, n
    best_score = float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        # Alternate row direction for a natural snake/boustrophedon wire
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts, cols, rows


def make_spline_wire(ms, pts):
    """
    Draw a spline through all light positions in snake order.
    Uses AddSpline via raw COM VARIANT arrays.
    """
    if len(pts) < 2:
        return None

    flat = []
    for (x, y) in pts:
        flat.extend([x, y, 0.0])

    point_array = win32com.client.VARIANT(
        pythoncom.VT_ARRAY | pythoncom.VT_R8,
        flat
    )
    tangent = win32com.client.VARIANT(
        pythoncom.VT_ARRAY | pythoncom.VT_R8,
        [1.0, 0.0, 0.0]
    )

    spline = ms.AddSpline(point_array, tangent, tangent)
    return spline


def add_label(ms, x, y, text, height, offset_y):
    """
    Add a centre-aligned MText label below the light insertion point.
    """
    label = ms.AddMText(
        APoint(x, y - offset_y, 0),
        height * 6,      # text box width
        text
    )
    label.Height          = height
    label.AttachmentPoint = 5   # middle-centre
    return label


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    """
    Ask how the circuit labels should be generated.

    Three modes
    ───────────
    A) Auto-increment  →  user types  "E20 1"
       Labels become E20.1, E20.2, E20.3 … (counter never resets between rectangles)

    B) Per-rectangle   →  user types  "E20 1 reset"
       Labels reset to E20.1 for each rectangle: useful when each room is a
       separate circuit.

    C) Fixed label     →  user types  "E20.2"
       Every light gets exactly that label.
    """
    print("\n── Circuit label configuration ──────────────────────────────")
    print("  Mode A — auto-increment across all rooms:")
    print("           type  <prefix> <start>           e.g.  E20 1")
    print("  Mode B — auto-increment, reset per room:")
    print("           type  <prefix> <start> reset     e.g.  E20 1 reset")
    print("  Mode C — same label on every light:")
    print("           type  <label>                    e.g.  E20.2")
    raw = input("  Your choice: ").strip()

    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start  = int(parts[1])
            reset  = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset   # (prefix, start, auto, reset_per_rect)
        except ValueError:
            pass
    # Fixed label
    return raw, None, False, False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # ── Connect to AutoCAD ──────────────────
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # ── Scan for rectangles ──────────────────
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed rectangles (4-vertex polylines) found in the drawing.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.3f}, {y0:.3f}) → ({x1:.3f}, {y1:.3f})"
              f"  size: {x1-x0:.3f} × {y1-y0:.3f}")

    # ── Ask which rectangles to use ──────────
    sel = input(
        "\nWhich rectangles to fill? "
        "(e.g. 1,2,3  or  'all'): "
    ).strip().lower()

    if sel == "all":
        chosen = list(range(len(rectangles)))
    else:
        chosen = []
        for s in sel.split(","):
            try:
                idx = int(s.strip()) - 1
                if 0 <= idx < len(rectangles):
                    chosen.append(idx)
            except ValueError:
                pass

    if not chosen:
        print("No valid selection. Exiting.")
        sys.exit(0)

    # ── Pick fixture ─────────────────────────
    fixture_name, block_name = pick_fixture()

    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found in the drawing.")
        print("Make sure the block is defined or insert it once manually first.")
        sys.exit(1)

    # ── Number of lights ─────────────────────
    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0:
                break
        except ValueError:
            pass
        print("  Please enter a positive integer.")

    # ── Circuit label ─────────────────────────
    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()

    # ── Wire / label toggles ──────────────────
    draw_wire  = input("\nDraw wire connection between lights? (y/n) [y]: ").strip().lower()
    draw_wire  = draw_wire != "n"
    draw_label = input("Add circuit label below each light?  (y/n) [y]: ").strip().lower()
    draw_label = draw_label != "n"

    # ── Prepare layers ────────────────────────
    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)   # 6 = magenta
    ensure_layer(doc, LABEL_LAYER, color=6)

    # ── Place everything ──────────────────────
    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []   # keep COM references alive

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

        rect_w   = x1 - x0
        rect_h   = y1 - y0
        text_h   = min(rect_w, rect_h) * LABEL_HEIGHT_RATIO
        label_dy = min(rect_w, rect_h) * LABEL_OFFSET_RATIO + text_h

        # Reset counter for this rectangle if mode B
        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Blocks ──────────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(
                APoint(px, py, 0),
                block_name,
                BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE,
                0.0
            )
            all_refs.append(ref)
            total += 1

        # ── Spline wire ──────────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = make_spline_wire(ms, pts)
            if wire:
                all_refs.append(wire)

        # ── Labels ───────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                if auto_inc:
                    txt = f"{label_prefix}.{label_index}"
                    label_index += 1
                else:
                    txt = label_prefix

                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'yes' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    # ── Save ──────────────────────────────────
    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing2.dwg

Found 4 rectangle(s):
  [1] (15.665, 26.595) → (162.235, 64.853)  size: 146.570 × 38.258
  [2] (1.601, 85.807) → (148.171, 124.065)  size: 146.570 × 38.258
  [3] (25.728, -19.818) → (172.298, 18.440)  size: 146.570 × 38.258
  [4] (-19.757, -81.306) → (126.813, -43.048)  size: 146.570 × 38.258

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label configuration ──────────────────────────────
  Mode A — auto-increment across all rooms:
           type  <prefix> <start>           e.g.  E20 1
  Mode B — auto-increment, reset per room:
           type  <prefix> <start> reset     e.g.  E20 1 reset
  Mode C — same label on every light:
           type  <label>                    e.g.  E20.2


TypeError: Cannot put win32com.client.VARIANT(8197, [1.0, 0.0, 0.0]) in VARIANT

In [31]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline with 4 vertices) in the drawing
2. For each rectangle, places a chosen light fixture in an evenly spaced grid
3. Draws a spline wire connecting all lights in reading order (row by row)
4. Adds a text label (circuit name) below each light
5. Grid is computed from the rectangle's own coordinates — no fixed margin needed

Requirements:
    pip install pyautocad pywin32

Usage:
    - Open your DWG in AutoCAD
    - Draw closed rectangles (RECTANG command) for each room/zone
    - Run:  python place_lights_auto.py
"""

import math
import sys
import pythoncom
import win32com.client

from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION — edit these
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"       # layer for spline wire
LABEL_LAYER = "ECLAIRAGE-LABEL"      # layer for text labels

# Layer name of rectangles to scan (None = scan all layers)
RECTANGLE_LAYER = None

# Block insertion scale
BLOCK_SCALE = 1.0

# Margin as a fraction of rectangle dimension (0.1 = 10%)
MARGIN_RATIO = 0.10

# Label text height — fraction of rectangle's smaller dimension
LABEL_HEIGHT_RATIO = 0.06

# Label offset below block insertion point (fraction of rectangle height)
LABEL_OFFSET_RATIO = 0.05


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(pline):
    try:
        if pline.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        coords = list(pline.Coordinates)
        step = 2 if pline.EntityName == "AcDbPolyline" else 3
        pts = [(coords[i], coords[i + 1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        return min(xs), min(ys), max(xs), max(ys)
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts, cols, rows) where pts is exactly n (x,y) points
    arranged in a grid that matches the rectangle's aspect ratio.

    Wire order: row by row, left→right, bottom→top so the spline
    snakes naturally through the room like in the reference image.
    """
    width  = x_max - x_min
    height = y_max - y_min

    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO

    ux0 = x_min + mx
    uy0 = y_min + my
    ux1 = x_max - mx
    uy1 = y_max - my

    uw = ux1 - ux0
    uh = uy1 - uy0

    # Find best cols × rows
    best_cols, best_rows = 1, n
    best_score = float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        # Alternate row direction for a natural snake/boustrophedon wire
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts, cols, rows


def make_spline_wire(ms, pts):
    """
    Draw a spline through all light positions in snake order.
    Uses AddSpline via raw COM VARIANT array for points.

    NOTE: Start/end tangents must be plain Python lists — pywin32
    wraps them automatically. Passing pre-built VARIANT objects for
    the tangents causes a TypeError ("Cannot put VARIANT in VARIANT").
    """
    if len(pts) < 2:
        return None

    flat = []
    for (x, y) in pts:
        flat.extend([x, y, 0.0])

    # Points array must be an explicit VARIANT of doubles
    point_array = win32com.client.VARIANT(
        pythoncom.VT_ARRAY | pythoncom.VT_R8,
        flat
    )

    # Tangents: plain lists — let pywin32 handle the wrapping
    tangent = [1.0, 0.0, 0.0]

    spline = ms.AddSpline(point_array, tangent, tangent)
    return spline


def add_label(ms, x, y, text, height, offset_y):
    """
    Add a centre-aligned MText label below the light insertion point.
    """
    label = ms.AddMText(
        APoint(x, y - offset_y, 0),
        height * 6,      # text box width
        text
    )
    label.Height          = height
    label.AttachmentPoint = 5   # middle-centre
    return label


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    """
    Ask how the circuit labels should be generated.

    Three modes
    ───────────
    A) Auto-increment  →  user types  "E20 1"
       Labels become E20.1, E20.2, E20.3 … (counter never resets between rectangles)

    B) Per-rectangle   →  user types  "E20 1 reset"
       Labels reset to E20.1 for each rectangle: useful when each room is a
       separate circuit.

    C) Fixed label     →  user types  "E20.2"
       Every light gets exactly that label.
    """
    print("\n── Circuit label configuration ──────────────────────────────")
    print("  Mode A — auto-increment across all rooms:")
    print("           type  <prefix> <start>           e.g.  E20 1")
    print("  Mode B — auto-increment, reset per room:")
    print("           type  <prefix> <start> reset     e.g.  E20 1 reset")
    print("  Mode C — same label on every light:")
    print("           type  <label>                    e.g.  E20.2")
    raw = input("  Your choice: ").strip()

    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start  = int(parts[1])
            reset  = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset   # (prefix, start, auto, reset_per_rect)
        except ValueError:
            pass
    # Fixed label
    return raw, None, False, False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # ── Connect to AutoCAD ──────────────────
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # ── Scan for rectangles ──────────────────
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed rectangles (4-vertex polylines) found in the drawing.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.3f}, {y0:.3f}) → ({x1:.3f}, {y1:.3f})"
              f"  size: {x1-x0:.3f} × {y1-y0:.3f}")

    # ── Ask which rectangles to use ──────────
    sel = input(
        "\nWhich rectangles to fill? "
        "(e.g. 1,2,3  or  'all'): "
    ).strip().lower()

    if sel == "all":
        chosen = list(range(len(rectangles)))
    else:
        chosen = []
        for s in sel.split(","):
            try:
                idx = int(s.strip()) - 1
                if 0 <= idx < len(rectangles):
                    chosen.append(idx)
            except ValueError:
                pass

    if not chosen:
        print("No valid selection. Exiting.")
        sys.exit(0)

    # ── Pick fixture ─────────────────────────
    fixture_name, block_name = pick_fixture()

    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found in the drawing.")
        print("Make sure the block is defined or insert it once manually first.")
        sys.exit(1)

    # ── Number of lights ─────────────────────
    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0:
                break
        except ValueError:
            pass
        print("  Please enter a positive integer.")

    # ── Circuit label ─────────────────────────
    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()

    # ── Wire / label toggles ──────────────────
    draw_wire  = input("\nDraw wire connection between lights? (y/n) [y]: ").strip().lower()
    draw_wire  = draw_wire != "n"
    draw_label = input("Add circuit label below each light?  (y/n) [y]: ").strip().lower()
    draw_label = draw_label != "n"

    # ── Prepare layers ────────────────────────
    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)   # 6 = magenta
    ensure_layer(doc, LABEL_LAYER, color=6)

    # ── Place everything ──────────────────────
    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []   # keep COM references alive

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

        rect_w   = x1 - x0
        rect_h   = y1 - y0
        text_h   = min(rect_w, rect_h) * LABEL_HEIGHT_RATIO
        label_dy = min(rect_w, rect_h) * LABEL_OFFSET_RATIO + text_h

        # Reset counter for this rectangle if mode B
        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Blocks ──────────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(
                APoint(px, py, 0),
                block_name,
                BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE,
                0.0
            )
            all_refs.append(ref)
            total += 1

        # ── Spline wire ──────────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = make_spline_wire(ms, pts)
            if wire:
                all_refs.append(wire)

        # ── Labels ───────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                if auto_inc:
                    txt = f"{label_prefix}.{label_index}"
                    label_index += 1
                else:
                    txt = label_prefix

                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'yes' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    # ── Save ──────────────────────────────────
    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing2.dwg

Found 5 rectangle(s):
  [1] (15.665, 26.595) → (162.235, 64.853)  size: 146.570 × 38.258
  [2] (1.601, 85.807) → (148.171, 124.065)  size: 146.570 × 38.258
  [3] (25.728, -19.818) → (172.298, 18.440)  size: 146.570 × 38.258
  [4] (-19.757, -81.306) → (126.813, -43.048)  size: 146.570 × 38.258
  [5] (-23.193, -156.611) → (123.376, -118.353)  size: 146.570 × 38.258

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label configuration ──────────────────────────────
  Mode A — auto-increment across all rooms:
           type  <prefix> <start>           e.g.  E20 1
  Mode B — auto-increment, reset per room:
           type  <prefix> <start> reset     e.g.  E20 1 reset
  Mode C — same label on every light:
           type  <label>                    e.g.  E20.2


TypeError: Cannot put win32com.client.VARIANT(8197, [6.120588630288296, -137.48172910898006, 0.0, 35.43450431342643, -137.48172910898006, 0.0, 64.74841999656456, -137.48172910898006, 0.0, 94.0623356797027, -137.48172910898006, 0.0]) in VARIANT

In [32]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline with 4 vertices) in the drawing
2. For each rectangle, places a chosen light fixture in an evenly spaced grid
3. Draws a spline wire connecting all lights in reading order (row by row)
4. Adds a text label (circuit name) below each light
5. Grid is computed from the rectangle's own coordinates — no fixed margin needed

Requirements:
    pip install pyautocad pywin32

Usage:
    - Open your DWG in AutoCAD
    - Draw closed rectangles (RECTANG command) for each room/zone
    - Run:  python place_lights_auto.py
"""

import math
import sys
import pythoncom
import win32com.client

from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION — edit these
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"       # layer for spline wire
LABEL_LAYER = "ECLAIRAGE-LABEL"      # layer for text labels

# Layer name of rectangles to scan (None = scan all layers)
RECTANGLE_LAYER = None

# Block insertion scale
BLOCK_SCALE = 1.0

# Margin as a fraction of rectangle dimension (0.1 = 10%)
MARGIN_RATIO = 0.10

# Label text height — fraction of rectangle's smaller dimension
LABEL_HEIGHT_RATIO = 0.06

# Label offset below block insertion point (fraction of rectangle height)
LABEL_OFFSET_RATIO = 0.05


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(pline):
    try:
        if pline.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        coords = list(pline.Coordinates)
        step = 2 if pline.EntityName == "AcDbPolyline" else 3
        pts = [(coords[i], coords[i + 1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        return min(xs), min(ys), max(xs), max(ys)
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts, cols, rows) where pts is exactly n (x,y) points
    arranged in a grid that matches the rectangle's aspect ratio.

    Wire order: row by row, left→right, bottom→top so the spline
    snakes naturally through the room like in the reference image.
    """
    width  = x_max - x_min
    height = y_max - y_min

    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO

    ux0 = x_min + mx
    uy0 = y_min + my
    ux1 = x_max - mx
    uy1 = y_max - my

    uw = ux1 - ux0
    uh = uy1 - uy0

    # Find best cols × rows
    best_cols, best_rows = 1, n
    best_score = float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        # Alternate row direction for a natural snake/boustrophedon wire
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts, cols, rows


def make_spline_wire(ms, pts):
    """
    Draw a spline through all light positions in snake order.
    Uses AddSpline via raw COM VARIANT array for points.

    NOTE: Start/end tangents must be plain Python lists — pywin32
    wraps them automatically. Passing pre-built VARIANT objects for
    the tangents causes a TypeError ("Cannot put VARIANT in VARIANT").
    """
    if len(pts) < 2:
        return None

    flat = []
    for (x, y) in pts:
        flat.extend([x, y, 0.0])

    # Pass all three as plain Python lists.
    # AutoCAD's late-bound COM dispatch wraps every argument into a
    # VARIANT itself, so pre-wrapping any of them causes the
    # "Cannot put VARIANT in VARIANT" double-wrap TypeError.
    tangent = [1.0, 0.0, 0.0]

    spline = ms.AddSpline(flat, tangent, tangent)
    return spline


def add_label(ms, x, y, text, height, offset_y):
    """
    Add a centre-aligned MText label below the light insertion point.
    """
    label = ms.AddMText(
        APoint(x, y - offset_y, 0),
        height * 6,      # text box width
        text
    )
    label.Height          = height
    label.AttachmentPoint = 5   # middle-centre
    return label


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    """
    Ask how the circuit labels should be generated.

    Three modes
    ───────────
    A) Auto-increment  →  user types  "E20 1"
       Labels become E20.1, E20.2, E20.3 … (counter never resets between rectangles)

    B) Per-rectangle   →  user types  "E20 1 reset"
       Labels reset to E20.1 for each rectangle: useful when each room is a
       separate circuit.

    C) Fixed label     →  user types  "E20.2"
       Every light gets exactly that label.
    """
    print("\n── Circuit label configuration ──────────────────────────────")
    print("  Mode A — auto-increment across all rooms:")
    print("           type  <prefix> <start>           e.g.  E20 1")
    print("  Mode B — auto-increment, reset per room:")
    print("           type  <prefix> <start> reset     e.g.  E20 1 reset")
    print("  Mode C — same label on every light:")
    print("           type  <label>                    e.g.  E20.2")
    raw = input("  Your choice: ").strip()

    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start  = int(parts[1])
            reset  = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset   # (prefix, start, auto, reset_per_rect)
        except ValueError:
            pass
    # Fixed label
    return raw, None, False, False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # ── Connect to AutoCAD ──────────────────
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # ── Scan for rectangles ──────────────────
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed rectangles (4-vertex polylines) found in the drawing.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.3f}, {y0:.3f}) → ({x1:.3f}, {y1:.3f})"
              f"  size: {x1-x0:.3f} × {y1-y0:.3f}")

    # ── Ask which rectangles to use ──────────
    sel = input(
        "\nWhich rectangles to fill? "
        "(e.g. 1,2,3  or  'all'): "
    ).strip().lower()

    if sel == "all":
        chosen = list(range(len(rectangles)))
    else:
        chosen = []
        for s in sel.split(","):
            try:
                idx = int(s.strip()) - 1
                if 0 <= idx < len(rectangles):
                    chosen.append(idx)
            except ValueError:
                pass

    if not chosen:
        print("No valid selection. Exiting.")
        sys.exit(0)

    # ── Pick fixture ─────────────────────────
    fixture_name, block_name = pick_fixture()

    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found in the drawing.")
        print("Make sure the block is defined or insert it once manually first.")
        sys.exit(1)

    # ── Number of lights ─────────────────────
    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0:
                break
        except ValueError:
            pass
        print("  Please enter a positive integer.")

    # ── Circuit label ─────────────────────────
    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()

    # ── Wire / label toggles ──────────────────
    draw_wire  = input("\nDraw wire connection between lights? (y/n) [y]: ").strip().lower()
    draw_wire  = draw_wire != "n"
    draw_label = input("Add circuit label below each light?  (y/n) [y]: ").strip().lower()
    draw_label = draw_label != "n"

    # ── Prepare layers ────────────────────────
    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)   # 6 = magenta
    ensure_layer(doc, LABEL_LAYER, color=6)

    # ── Place everything ──────────────────────
    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []   # keep COM references alive

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

        rect_w   = x1 - x0
        rect_h   = y1 - y0
        text_h   = min(rect_w, rect_h) * LABEL_HEIGHT_RATIO
        label_dy = min(rect_w, rect_h) * LABEL_OFFSET_RATIO + text_h

        # Reset counter for this rectangle if mode B
        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Blocks ──────────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(
                APoint(px, py, 0),
                block_name,
                BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE,
                0.0
            )
            all_refs.append(ref)
            total += 1

        # ── Spline wire ──────────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = make_spline_wire(ms, pts)
            if wire:
                all_refs.append(wire)

        # ── Labels ───────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                if auto_inc:
                    txt = f"{label_prefix}.{label_index}"
                    label_index += 1
                else:
                    txt = label_prefix

                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'yes' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    # ── Save ──────────────────────────────────
    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing2.dwg

Found 6 rectangle(s):
  [1] (15.665, 26.595) → (162.235, 64.853)  size: 146.570 × 38.258
  [2] (1.601, 85.807) → (148.171, 124.065)  size: 146.570 × 38.258
  [3] (25.728, -19.818) → (172.298, 18.440)  size: 146.570 × 38.258
  [4] (-19.757, -81.306) → (126.813, -43.048)  size: 146.570 × 38.258
  [5] (-23.193, -156.611) → (123.376, -118.353)  size: 146.570 × 38.258
  [6] (-57.734, -217.151) → (88.836, -178.893)  size: 146.570 × 38.258

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label configuration ──────────────────────────────
  Mode A — auto-increment across all rooms:
           type  <prefix> <start>           e.g.  E20 1
  Mode B — auto-increment, reset per room:
           type  <prefix> <start> reset     e.g.  E20 1 reset
  Mode C — same label on every light:
 

COMError: (-2147352567, 'Exception occurred.', (None, None, None, 0, None))

In [25]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline with 4 vertices) in the drawing
2. For each rectangle, places a chosen light fixture in an evenly spaced grid
3. Draws a spline wire connecting all lights in reading order (row by row)
4. Adds a text label (circuit name) below each light
5. Grid is computed from the rectangle's own coordinates — no fixed margin needed

Requirements:
    pip install pyautocad pywin32

Usage:
    - Open your DWG in AutoCAD
    - Draw closed rectangles (RECTANG command) for each room/zone
    - Run:  python place_lights_auto.py
"""

import math
import sys
import pythoncom
import win32com.client

from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION — edit these
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"       # layer for spline wire
LABEL_LAYER = "ECLAIRAGE-LABEL"      # layer for text labels

# Layer name of rectangles to scan (None = scan all layers)
RECTANGLE_LAYER = None

# Block insertion scale
BLOCK_SCALE = 1.0

# Margin as a fraction of rectangle dimension (0.1 = 10%)
MARGIN_RATIO = 0.001

# Label text height — fraction of rectangle's smaller dimension
LABEL_HEIGHT_RATIO = 0.04

# Label offset below block insertion point (fraction of rectangle height)
LABEL_OFFSET_RATIO = 0.01


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(pline):
    try:
        if pline.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        coords = list(pline.Coordinates)
        step = 2 if pline.EntityName == "AcDbPolyline" else 3
        pts = [(coords[i], coords[i + 1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        return min(xs), min(ys), max(xs), max(ys)
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts, cols, rows) where pts is exactly n (x,y) points
    arranged in a grid that matches the rectangle's aspect ratio.

    Wire order: row by row, left→right, bottom→top so the spline
    snakes naturally through the room like in the reference image.
    """
    width  = x_max - x_min
    height = y_max - y_min

    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO

    ux0 = x_min + mx
    uy0 = y_min + my
    ux1 = x_max - mx
    uy1 = y_max - my

    uw = ux1 - ux0
    uh = uy1 - uy0

    # Find best cols × rows
    best_cols, best_rows = 1, n
    best_score = float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        # Alternate row direction for a natural snake/boustrophedon wire
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts, cols, rows


def make_spline_wire(ms, pts):
    """Spline wire — temporarily disabled, returns None."""
    return None


def add_label(ms, x, y, text, height, offset_y):
    """
    Add a centre-aligned MText label below the light insertion point.
    """
    label = ms.AddMText(
        APoint(x, y - offset_y, 0),
        height * 6,      # text box width
        text
    )
    label.Height          = height
    label.AttachmentPoint = 1  # middle-centre
    return label


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    """
    Ask how the circuit labels should be generated.

    Three modes
    ───────────
    A) Auto-increment  →  user types  "E20 1"
       Labels become E20.1, E20.2, E20.3 … (counter never resets between rectangles)

    B) Per-rectangle   →  user types  "E20 1 reset"
       Labels reset to E20.1 for each rectangle: useful when each room is a
       separate circuit.

    C) Fixed label     →  user types  "E20.2"
       Every light gets exactly that label.
    """
    print("\n── Circuit label configuration ──────────────────────────────")
    print("  Mode A — auto-increment across all rooms:")
    print("           type  <prefix> <start>           e.g.  E20 1")
    print("  Mode B — auto-increment, reset per room:")
    print("           type  <prefix> <start> reset     e.g.  E20 1 reset")
    print("  Mode C — same label on every light:")
    print("           type  <label>                    e.g.  E20.2")
    raw = input("  Your choice: ").strip()

    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start  = int(parts[1])
            reset  = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset   # (prefix, start, auto, reset_per_rect)
        except ValueError:
            pass
    # Fixed label
    return raw, None, False, False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # ── Connect to AutoCAD ──────────────────
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # ── Scan for rectangles ──────────────────
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed rectangles (4-vertex polylines) found in the drawing.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.3f}, {y0:.3f}) → ({x1:.3f}, {y1:.3f})"
              f"  size: {x1-x0:.3f} × {y1-y0:.3f}")

    # ── Ask which rectangles to use ──────────
    sel = input(
        "\nWhich rectangles to fill? "
        "(e.g. 1,2,3  or  'all'): "
    ).strip().lower()

    if sel == "all":
        chosen = list(range(len(rectangles)))
    else:
        chosen = []
        for s in sel.split(","):
            try:
                idx = int(s.strip()) - 1
                if 0 <= idx < len(rectangles):
                    chosen.append(idx)
            except ValueError:
                pass

    if not chosen:
        print("No valid selection. Exiting.")
        sys.exit(0)

    # ── Pick fixture ─────────────────────────
    fixture_name, block_name = pick_fixture()

    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found in the drawing.")
        print("Make sure the block is defined or insert it once manually first.")
        sys.exit(1)

    # ── Number of lights ─────────────────────
    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0:
                break
        except ValueError:
            pass
        print("  Please enter a positive integer.")

    # ── Circuit label ─────────────────────────
    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()

    # ── Wire / label toggles ──────────────────
    draw_wire  = input("\nDraw wire connection between lights? (y/n) [y]: ").strip().lower()
    draw_wire  = draw_wire != "n"
    draw_label = input("Add circuit label below each light?  (y/n) [y]: ").strip().lower()
    draw_label = draw_label != "n"

    # ── Prepare layers ────────────────────────
    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)   # 6 = magenta
    ensure_layer(doc, LABEL_LAYER, color=6)

    # ── Place everything ──────────────────────
    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []   # keep COM references alive

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

        rect_w   = x1 - x0
        rect_h   = y1 - y0
        text_h   = min(rect_w, rect_h) * LABEL_HEIGHT_RATIO
        label_dy = min(rect_w, rect_h) * LABEL_OFFSET_RATIO + text_h

        # Reset counter for this rectangle if mode B
        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Blocks ──────────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(
                APoint(px, py, 0),
                block_name,
                BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE,
                0.0
            )
            all_refs.append(ref)
            total += 1

        # ── Spline wire ──────────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = make_spline_wire(ms, pts)
            if wire:
                all_refs.append(wire)

        # ── Labels ───────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                if auto_inc:
                    txt = f"{label_prefix}.{label_index}"
                    label_index += 1
                else:
                    txt = label_prefix

                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'yes' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    # ── Save ──────────────────────────────────
    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg

Found 2 rectangle(s):
  [1] (413.158, 2139.204) → (39118.163, 19654.844)  size: 38705.005 × 17515.640
  [2] (-2023.445, 25525.295) → (44549.333, 47620.824)  size: 46572.779 × 22095.529
No valid selection. Exiting.


SystemExit: 0

In [27]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline) in the drawing
2. Places chosen light fixtures in an evenly spaced grid
3. Draws a snake-pattern spline wire connecting all lights row by row
4. Adds black circuit labels below each light

Requirements:
    pip install pyautocad pywin32
"""

import math
import sys
import array
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"

RECTANGLE_LAYER  = None
BLOCK_SCALE      = 1.0
MARGIN_RATIO     = 0.001
LABEL_HEIGHT_RATIO = 0.04
LABEL_OFFSET_RATIO = 0.01

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    """Use GetBoundingBox — reliable, avoids broken COM variant from .Coordinates."""
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        min_pt, max_pt = entity.GetBoundingBox()
        x0, y0 = float(min_pt[0]), float(min_pt[1])
        x1, y1 = float(max_pt[0]), float(max_pt[1])
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Returns (pts_in_order, cols, rows).
    pts_in_order: snake path — row 0 left→right, row 1 right→left, etc.
    This matches the wire pattern in the reference image.
    """
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw, uh = ux1 - ux0, uy1 - uy0

    # Best cols×rows for aspect ratio
    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        # Alternate direction — boustrophedon snake pattern
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows


def draw_snake_wire(ms, pts):
    """
    Draw a spline through all light positions in snake order.
    The spline passes through each light center, creating smooth
    curves that match the reference image.
    """
    if len(pts) < 2:
        return None

    # Build flat array of 3D points for AddSpline: [x0,y0,z0, x1,y1,z1, ...]
    flat = []
    for (px, py) in pts:
        flat.extend([px, py, 0.0])

    pt_array = array.array('d', flat)

    # Start and end tangent vectors (horizontal, pointing in wire direction)
    start_tan = array.array('d', [1.0, 0.0, 0.0])
    end_tan   = array.array('d', [1.0, 0.0, 0.0])

    try:
        spline = ms.AddSpline(pt_array, start_tan, end_tan)
        return spline
    except Exception as e:
        print(f"    Warning: could not draw wire — {e}")
        return None


def add_label(ms, x, y, text, height, offset_y):
    """Centre-aligned MText label below the light, color = black (256 = ByLayer, 7 = white/black)."""
    try:
        lbl = ms.AddMText(APoint(x, y - offset_y, 0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5   # middle-centre
        lbl.Color           = 0   # 0 = ByBlock; use 256 for ByLayer; 7 = black/white
                                  # For true black: set to 7 (AutoCAD color index 7)
        lbl.Color           = 7   # BLACK (color index 7 in AutoCAD)
        return lbl
    except Exception as e:
        print(f"    Warning: label error — {e}")
        return None


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    print("\n── Circuit label ─────────────────────────────────────────")
    print("  Auto-increment:          E20 1          → E20.1, E20.2 …")
    print("  Auto-increment + reset:  E20 1 reset    → resets per room")
    print("  Fixed label:             E20.2          → same on all lights")
    raw = input("  Your choice: ").strip()
    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start = int(parts[1])
            reset = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset
        except ValueError:
            pass
    return raw, None, False, False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # Scan rectangles
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed rectangles found. Use RECTANG to draw room outlines.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"   {x1-x0:.1f} × {y1-y0:.1f} units")

    sel = input("\nWhich rectangles to fill? (e.g. 1,2  or  'all'): ").strip().lower()
    chosen = list(range(len(rectangles))) if sel == "all" else [
        int(s.strip()) - 1 for s in sel.split(",")
        if s.strip().isdigit() and 0 <= int(s.strip()) - 1 < len(rectangles)
    ]
    if not chosen:
        print("No valid selection."); sys.exit(0)

    fixture_name, block_name = pick_fixture()
    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found. Insert it manually once first.")
        sys.exit(1)

    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0: break
        except ValueError:
            pass
        print("  Enter a positive integer.")

    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()

    draw_wire  = input("\nDraw wire between lights? (y/n) [y]: ").strip().lower() != "n"
    draw_label = input("Add circuit labels?         (y/n) [y]: ").strip().lower() != "n"

    # Prepare layers
    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)    # magenta wire like the reference
    ensure_layer(doc, LABEL_LAYER, color=7)    # black labels

    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

        rect_min = min(x1 - x0, y1 - y0)
        text_h   = rect_min * LABEL_HEIGHT_RATIO
        label_dy = rect_min * LABEL_OFFSET_RATIO + text_h

        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Place blocks ────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(APoint(px, py, 0), block_name,
                                 BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
            all_refs.append(ref)
            total += 1

        # ── Draw snake wire ─────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = draw_snake_wire(ms, pts)
            if wire:
                all_refs.append(wire)

        # ── Labels ──────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                txt = f"{label_prefix}.{label_index}" if auto_inc else label_prefix
                if auto_inc:
                    label_index += 1
                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                if lbl:
                    all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'yes' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    # Commit
    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg
No closed rectangles found. Use RECTANG to draw room outlines.


SystemExit: 0

In [29]:
"""
diagnose.py
-----------
Prints every entity in ModelSpace so we can see exactly
what entity names and types AutoCAD is returning.
"""
from pyautocad import Autocad

acad = Autocad(create_if_not_exists=False)
doc  = acad.doc
ms   = acad.model

print(f"Connected to: {doc.Name}")
print(f"\nAll entities in ModelSpace:")
print(f"{'#':<4} {'EntityName':<25} {'Layer':<20} {'ObjectName'}")
print("-" * 75)

for i, entity in enumerate(ms, 1):
    try:
        ename = entity.EntityName
    except:
        ename = "N/A"
    try:
        layer = entity.Layer
    except:
        layer = "N/A"
    try:
        oname = entity.ObjectName
    except:
        oname = "N/A"

    print(f"{i:<4} {ename:<25} {layer:<20} {oname}")

    # If it looks like a polyline, dump more info
    if "Poly" in ename or "Poly" in oname:
        try:
            min_pt, max_pt = entity.GetBoundingBox()
            print(f"       BoundingBox: ({float(min_pt[0]):.2f}, {float(min_pt[1]):.2f})"
                  f" → ({float(max_pt[0]):.2f}, {float(max_pt[1]):.2f})")
        except Exception as e:
            print(f"       BoundingBox ERROR: {e}")
        try:
            coords = list(entity.Coordinates)
            print(f"       Coordinates raw ({len(coords)} values): {coords[:12]}...")
        except Exception as e:
            print(f"       Coordinates ERROR: {e}")
        try:
            print(f"       Closed: {entity.Closed}")
        except Exception as e:
            print(f"       Closed ERROR: {e}")

print("\nDone.")

Connected to: Drawing1.dwg

All entities in ModelSpace:
#    EntityName                Layer                ObjectName
---------------------------------------------------------------------------
1    AcDbBlockReference        0                    AcDbBlockReference
2    AcDbPolyline              0                    AcDbPolyline
       BoundingBox ERROR: (-2147352562, 'Invalid number of parameters.', (None, None, None, 0, None))
       Coordinates raw (8 values): [413.15783572217333, 19654.843753955967, 39118.162815877906, 19654.843753955967, 39118.162815877906, 2139.203633302124, 413.15783572217333, 2139.203633302124]...
       Closed: True
3    AcDbBlockReference        ECLAIRAGE            AcDbBlockReference
4    AcDbBlockReference        0                    AcDbBlockReference

Done.


In [31]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline) in the drawing
2. Places chosen light fixtures in an evenly spaced grid
3. Draws a snake-pattern spline wire connecting all lights row by row
4. Adds black circuit labels below each light

Requirements:
    pip install pyautocad pywin32
"""

import math
import sys
import array
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"

RECTANGLE_LAYER    = None
BLOCK_SCALE        = 1.0
MARGIN_RATIO       = 0.001
LABEL_HEIGHT_RATIO = 0.04
LABEL_OFFSET_RATIO = 0.01

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    """
    Read bounds directly from .Coordinates (flat list of x,y pairs).
    Works even when GetBoundingBox raises 'Invalid number of parameters'.
    Returns (x_min, y_min, x_max, y_max) or None.
    """
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None

        # Must be closed
        if not entity.Closed:
            return None

        coords = list(entity.Coordinates)  # [x0,y0, x1,y1, x2,y2, x3,y3]

        # LWPolyline = 2D coords (step 2), 2dPolyline = 3D (step 3)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]

        # Must be exactly 4 vertices (rectangle)
        if len(pts) != 4:
            return None

        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)

        # Reject degenerate shapes
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None

        return x0, y0, x1, y1

    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Returns (pts_in_snake_order, cols, rows).
    Snake: row 0 left→right, row 1 right→left, etc.
    """
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    # Best cols×rows for aspect ratio
    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()   # alternate direction = snake
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows


def draw_snake_wire(ms, pts):
    """Spline through all lights in snake order."""
    if len(pts) < 2:
        return None
    flat = []
    for (px, py) in pts:
        flat.extend([px, py, 0.0])
    pt_array  = array.array('d', flat)
    start_tan = array.array('d', [1.0, 0.0, 0.0])
    end_tan   = array.array('d', [1.0, 0.0, 0.0])
    try:
        return ms.AddSpline(pt_array, start_tan, end_tan)
    except Exception as e:
        print(f"    Warning: wire error — {e}")
        return None


def add_label(ms, x, y, text, height, offset_y):
    """Black centre-aligned MText label below the light."""
    try:
        lbl = ms.AddMText(APoint(x, y - offset_y, 0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5   # middle-centre
        lbl.Color           = 7   # AutoCAD color 7 = black
        return lbl
    except Exception as e:
        print(f"    Warning: label error — {e}")
        return None


def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    print("\n── Circuit label ─────────────────────────────────────────")
    print("  Auto-increment:          E20 1          → E20.1, E20.2 …")
    print("  Auto-increment + reset:  E20 1 reset    → resets per room")
    print("  Fixed label:             E20.2          → same on all lights")
    raw   = input("  Your choice: ").strip()
    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start = int(parts[1])
            reset = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset
        except ValueError:
            pass
    return raw, None, False, False


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # Scan rectangles
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed 4-vertex polylines found. Draw room outlines with RECTANG.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"   {x1-x0:.1f} × {y1-y0:.1f} units")

    sel = input("\nWhich rectangles to fill? (e.g. 1,2  or  'all'): ").strip().lower()
    chosen = list(range(len(rectangles))) if sel == "all" else [
        int(s.strip()) - 1 for s in sel.split(",")
        if s.strip().isdigit() and 0 <= int(s.strip()) - 1 < len(rectangles)
    ]
    if not chosen:
        print("No valid selection."); sys.exit(0)

    fixture_name, block_name = pick_fixture()
    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found. Insert it manually once first.")
        sys.exit(1)

    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0: break
        except ValueError:
            pass
        print("  Enter a positive integer.")

    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()
    draw_wire  = input("\nDraw wire between lights? (y/n) [y]: ").strip().lower() != "n"
    draw_label = input("Add circuit labels?         (y/n) [y]: ").strip().lower() != "n"

    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)
    ensure_layer(doc, LABEL_LAYER, color=7)

    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

        rect_min = min(x1 - x0, y1 - y0)
        text_h   = rect_min * LABEL_HEIGHT_RATIO
        label_dy = rect_min * LABEL_OFFSET_RATIO + text_h

        if auto_inc and reset_per_rect:
            label_index = label_start

        # Blocks
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(APoint(px, py, 0), block_name,
                                 BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
            all_refs.append(ref)
            total += 1

        # Wire
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = draw_snake_wire(ms, pts)
            if wire:
                all_refs.append(wire)

        # Labels
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                txt = f"{label_prefix}.{label_index}" if auto_inc else label_prefix
                if auto_inc:
                    label_index += 1
                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                if lbl:
                    all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'yes' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg

Found 2 rectangle(s):
  [1] (413.2, 2139.2) → (39118.2, 19654.8)   38705.0 × 17515.6 units
  [2] (413.2, 36360.0) → (39118.2, 53875.6)   38705.0 × 17515.6 units

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label ─────────────────────────────────────────
  Auto-increment:          E20 1          → E20.1, E20.2 …
  Auto-increment + reset:  E20 1 reset    → resets per room
  Fixed label:             E20.2          → same on all lights
  Rectangle [2]: 10 lights  grid 5×2  wire=yes  labels=yes

✓ Done. 10 light(s) placed and saved.


In [2]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline) in the drawing
2. Places chosen light fixtures in an evenly spaced grid
3. Draws 3-point ARCS between consecutive lights
   - The midpoint of each arc is offset perpendicular to the chord
   - ARC_BULGE_RATIO controls curvature (positive = bulge up/left)
4. Adds circuit labels below each light

Requirements:
    pip install pyautocad pywin32
"""

import math
import sys
import array
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"

RECTANGLE_LAYER    = None
BLOCK_SCALE        = 1.0
MARGIN_RATIO       = 0.001
LABEL_HEIGHT_RATIO = 0.04
LABEL_OFFSET_RATIO = 0.01

# Arc curvature: fraction of chord length used as perpendicular offset.
# 0.0  = straight line (degenerate arc)
# 0.25 = gentle curve  (default, matches your screenshot)
# 0.5  = tighter arc
# Negative values flip the arc to the other side.
ARC_BULGE_RATIO = 0.15  # default value is 0.25


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts_in_snake_order, cols, rows).
    Snake: row 0 left→right, row 1 right→left, etc.
    """
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows


# ─────────────────────────────────────────────
# ARC DRAWING
# ─────────────────────────────────────────────

def arc_midpoint(p1, p2, bulge_ratio):
    """
    Compute the midpoint that lies ON the arc between p1 and p2.

    Method
    ------
    1. Find chord midpoint M = (p1+p2)/2
    2. Compute perpendicular unit vector to the chord
    3. Offset M by  bulge_ratio × chord_length  along that perpendicular
       → this is the "through" point passed to AddArc3P

    bulge_ratio > 0 : arc bulges to the LEFT of the p1→p2 direction
    bulge_ratio < 0 : arc bulges to the RIGHT
    """
    mx = (p1[0] + p2[0]) / 2.0
    my = (p1[1] + p2[1]) / 2.0

    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    chord = math.hypot(dx, dy)

    if chord < 1e-9:
        return (mx, my)

    # Perpendicular unit vector (rotate 90° counter-clockwise)
    px = -dy / chord
    py =  dx / chord

    offset = bulge_ratio * chord
    return (mx + px * offset, my + py * offset)


def draw_arc_3p(ms, p1, mid, p2):
    """
    Draw a 3-point arc in AutoCAD model space.
    AutoCAD COM: AddArc(center, radius, startAngle, endAngle) — NOT 3-point.
    We must convert 3 points → center + radius + angles ourselves,
    then call ms.AddArc.

    Three points uniquely define a circle; we find it, then draw the
    minor arc from p1 to p2 passing through mid.
    """
    ax, ay = p1
    bx, by = mid
    cx, cy = p2

    # Circumcircle of (p1, mid, p2)
    D = 2 * (ax * (by - cy) + bx * (cy - ay) + cx * (ay - by))
    if abs(D) < 1e-12:
        # Points are collinear — fall back to a polyline segment
        flat = array.array('d', [ax, ay, 0.0, cx, cy, 0.0])
        try:
            ms.Add3DPoly(flat)
        except Exception:
            pass
        return None

    ux = ((ax**2 + ay**2) * (by - cy) +
          (bx**2 + by**2) * (cy - ay) +
          (cx**2 + cy**2) * (ay - by)) / D

    uy = ((ax**2 + ay**2) * (cx - bx) +
          (bx**2 + by**2) * (ax - cx) +
          (cx**2 + cy**2) * (bx - ax)) / D

    radius = math.hypot(ax - ux, ay - uy)

    # Angles from center to p1 and p2
    start_ang = math.atan2(ay - uy, ax - ux)
    end_ang   = math.atan2(cy - uy, cx - ux)

    # Determine which arc direction passes through mid
    mid_ang   = math.atan2(by - uy, bx - ux)

    def angle_between(a, start, end):
        """True if angle a lies on the counter-clockwise arc start→end."""
        a     = a     % (2 * math.pi)
        start = start % (2 * math.pi)
        end   = end   % (2 * math.pi)
        if start <= end:
            return start <= a <= end
        else:
            return a >= start or a <= end

    # If mid is NOT on the CCW arc p1→p2, swap start/end to get the CW arc
    if not angle_between(mid_ang, start_ang, end_ang):
        start_ang, end_ang = end_ang, start_ang

    try:
        arc = ms.AddArc(APoint(ux, uy, 0), radius,
                        math.degrees(start_ang),
                        math.degrees(end_ang))
        return arc
    except Exception as e:
        print(f"    Warning: arc error — {e}")
        return None


def draw_wire_arcs(ms, pts, bulge_ratio):
    """
    Draw one 3-point arc for every consecutive pair of lights.
    Returns list of arc objects (keep alive to avoid COM GC).
    """
    arcs = []
    for i in range(len(pts) - 1):
        p1  = pts[i]
        p2  = pts[i + 1]
        mid = arc_midpoint(p1, p2, bulge_ratio)
        arc = draw_arc_3p(ms, p1, mid, p2)
        if arc:
            arcs.append(arc)
    return arcs


# ─────────────────────────────────────────────
# LABEL
# ─────────────────────────────────────────────

def add_label(ms, x, y, text, height, offset_y):
    try:
        lbl = ms.AddMText(APoint(x, y - offset_y, 0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5   # middle-centre
        lbl.Color           = 7   # black
        return lbl
    except Exception as e:
        print(f"    Warning: label error — {e}")
        return None


# ─────────────────────────────────────────────
# USER INPUT HELPERS
# ─────────────────────────────────────────────

def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    print("\n── Circuit label ─────────────────────────────────────────")
    print("  Auto-increment:          E20 1          → E20.1, E20.2 …")
    print("  Auto-increment + reset:  E20 1 reset    → resets per room")
    print("  Fixed label:             E20.2          → same on all lights")
    raw   = input("  Your choice: ").strip()
    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start = int(parts[1])
            reset = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset
        except ValueError:
            pass
    return raw, None, False, False


def ask_bulge():
    """
    Ask the user for the arc curvature.
    Shows the default and explains what the value means.
    """
    print(f"\n── Arc curvature (bulge ratio) ───────────────────────────")
    print(f"  Controls how curved the wire is between each pair of lights.")
    print(f"  0.0  = straight line      0.25 = gentle arc (default)")
    print(f"  0.5  = tighter arc        negative = flip arc direction")
    raw = input(f"  Enter value [default {ARC_BULGE_RATIO}]: ").strip()
    if raw == "":
        return ARC_BULGE_RATIO
    try:
        val = float(raw)
        return val
    except ValueError:
        print(f"  Invalid — using default {ARC_BULGE_RATIO}")
        return ARC_BULGE_RATIO


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    # Scan rectangles
    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed 4-vertex polylines found. Draw room outlines with RECTANG.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"   {x1-x0:.1f} × {y1-y0:.1f} units")

    sel = input("\nWhich rectangles to fill? (e.g. 1,2  or  'all'): ").strip().lower()
    chosen = list(range(len(rectangles))) if sel == "all" else [
        int(s.strip()) - 1 for s in sel.split(",")
        if s.strip().isdigit() and 0 <= int(s.strip()) - 1 < len(rectangles)
    ]
    if not chosen:
        print("No valid selection."); sys.exit(0)

    fixture_name, block_name = pick_fixture()
    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found. Insert it manually once first.")
        sys.exit(1)

    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0: break
        except ValueError:
            pass
        print("  Enter a positive integer.")

    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()

    draw_wire  = input("\nDraw wire between lights? (y/n) [y]: ").strip().lower() != "n"
    bulge      = ask_bulge() if draw_wire else 0.0
    draw_label = input("Add circuit labels?         (y/n) [y]: ").strip().lower() != "n"

    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)   # magenta
    ensure_layer(doc, LABEL_LAYER, color=7)   # black

    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

        rect_min = min(x1 - x0, y1 - y0)
        text_h   = rect_min * LABEL_HEIGHT_RATIO
        label_dy = rect_min * LABEL_OFFSET_RATIO + text_h

        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Blocks ──────────────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(APoint(px, py, 0), block_name,
                                 BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
            all_refs.append(ref)
            total += 1

        # ── Arcs ─────────────────────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            arcs = draw_wire_arcs(ms, pts, bulge)
            all_refs.extend(arcs)
            print(f"    Drew {len(arcs)} arc(s) with bulge={bulge:.3f}")

        # ── Labels ───────────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                txt = f"{label_prefix}.{label_index}" if auto_inc else label_prefix
                if auto_inc:
                    label_index += 1
                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                if lbl:
                    all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'arc' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg

Found 2 rectangle(s):
  [1] (21323.9, 10669.2) → (38908.6, 16890.0)   17584.7 × 6220.8 units
  [2] (21323.9, 23021.9) → (38908.6, 29242.7)   17584.7 × 6220.8 units

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label ─────────────────────────────────────────
  Auto-increment:          E20 1          → E20.1, E20.2 …
  Auto-increment + reset:  E20 1 reset    → resets per room
  Fixed label:             E20.2          → same on all lights

── Arc curvature (bulge ratio) ───────────────────────────
  Controls how curved the wire is between each pair of lights.
  0.0  = straight line      0.25 = gentle arc (default)
  0.5  = tighter arc        negative = flip arc direction
    Drew 5 arc(s) with bulge=0.150
  Rectangle [2]: 6 lights  grid 3×2  wire=arc  labels=yes

✓ Done.

In [6]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline) in the drawing
2. Places chosen light fixtures in an evenly spaced grid
3. Draws arcs between consecutive lights using grid coordinates:
   - Center = midpoint between p1 and p2
   - Radius = half the distance between p1 and p2
   - Start angle = angle from center to p1
   - End angle   = angle from center to p2
   - Bulge shifts the center perpendicularly to control curvature
4. Adds circuit labels below each light

Requirements:
    pip install pyautocad pywin32
"""

import math
import sys
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"

RECTANGLE_LAYER    = None
BLOCK_SCALE        = 1.0
MARGIN_RATIO       = 0.001
LABEL_HEIGHT_RATIO = 0.04
LABEL_OFFSET_RATIO = 0.01

# Default arc curvature:
# 0.0 = semicircle (center exactly between the two lights)
# positive = shift center perpendicularly → tighter arc
# negative = flip to other side
ARC_BULGE_RATIO = 0.0


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts_in_snake_order, cols, rows, cell_w, cell_h).
    Snake: row 0 left→right, row 1 right→left, etc.
    We also return cell dimensions so the arc radius can be
    derived directly from the grid spacing.
    """
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows, cell_w, cell_h


# ─────────────────────────────────────────────
# ARC — grid-based, exact start/end on lights
# ─────────────────────────────────────────────

def draw_arc_between(ms, p1, p2, bulge_ratio):
    """
    Draw one arc whose endpoints are exactly p1 and p2.

    Geometry
    --------
                    perp offset (bulge)
                          ↑
    p1 ────── chord_mid ────── p2
               (arc center)

    1. arc_center = midpoint(p1, p2) shifted perpendicularly
       by  bulge_ratio × half_chord  (0 = no shift = semicircle)
    2. radius  = distance(arc_center, p1)  [= distance to p2 by symmetry]
    3. start_angle = atan2(p1 - arc_center)
    4. end_angle   = atan2(p2 - arc_center)
    5. Choose CCW or CW so the arc stays on the correct side.

    Because the center is computed from p1 and p2 the arc is
    GUARANTEED to start exactly at p1 and end exactly at p2.
    """
    x1, y1 = p1
    x2, y2 = p2

    # Chord vector and length
    dx   = x2 - x1
    dy   = y2 - y1
    chord = math.hypot(dx, dy)
    if chord < 1e-9:
        return None

    # Chord midpoint
    mx = (x1 + x2) / 2.0
    my = (y1 + y2) / 2.0

    # Perpendicular unit vector (90° CCW from chord direction)
    px = -dy / chord
    py =  dx / chord

    # Shift center perpendicularly — bulge_ratio=0 → center = chord midpoint
    # which gives a perfect semicircle
    shift = bulge_ratio * (chord / 2.0)
    cx = mx + px * shift
    cy = my + py * shift

    # Radius: distance from shifted center to p1 (= distance to p2)
    radius = math.hypot(x1 - cx, y1 - cy)
    if radius < 1e-9:
        return None

    # Angles from arc center to each light
    ang1 = math.degrees(math.atan2(y1 - cy, x1 - cx))
    ang2 = math.degrees(math.atan2(y2 - cy, x2 - cx))

    # We want the SHORT arc (the one that stays between the two lights).
    # AutoCAD AddArc always draws CCW from start_angle to end_angle.
    # Pick the ordering that gives the smaller sweep angle.
    def ccw_sweep(a, b):
        """CCW angular sweep from a to b, result in [0, 360)."""
        return (b - a) % 360.0

    sweep_fwd = ccw_sweep(ang1, ang2)   # CCW: ang1 → ang2
    sweep_rev = ccw_sweep(ang2, ang1)   # CCW: ang2 → ang1 (= CW version)

    if sweep_fwd <= sweep_rev:
        start_ang, end_ang = ang1, ang2
    else:
        start_ang, end_ang = ang2, ang1

    try:
        arc = ms.AddArc(APoint(cx, cy, 0), radius, start_ang, end_ang)
        return arc
    except Exception as e:
        print(f"    Warning: arc error — {e}")
        return None


def draw_wire_arcs(ms, pts, bulge_ratio):
    """One arc per consecutive pair of lights in snake order."""
    arcs = []
    for i in range(len(pts) - 1):
        arc = draw_arc_between(ms, pts[i], pts[i + 1], bulge_ratio)
        if arc:
            arcs.append(arc)
    return arcs


# ─────────────────────────────────────────────
# LABEL
# ─────────────────────────────────────────────

def add_label(ms, x, y, text, height, offset_y):
    try:
        lbl = ms.AddMText(APoint(x, y - offset_y, 0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5
        lbl.Color           = 7
        return lbl
    except Exception as e:
        print(f"    Warning: label error — {e}")
        return None


# ─────────────────────────────────────────────
# USER INPUT
# ─────────────────────────────────────────────

def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    print("\n── Circuit label ─────────────────────────────────────────")
    print("  Auto-increment:          E20 1          → E20.1, E20.2 …")
    print("  Auto-increment + reset:  E20 1 reset    → resets per room")
    print("  Fixed label:             E20.2          → same on all lights")
    raw   = input("  Your choice: ").strip()
    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start = int(parts[1])
            reset = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset
        except ValueError:
            pass
    return raw, None, False, False


def ask_bulge():
    print(f"\n── Arc curvature (bulge ratio) ───────────────────────────")
    print(f"  0.0  = semicircle between the two lights (default)")
    print(f"  0.5  = flatter arc (center shifted, smaller sweep)")
    print(f"  -0.5 = arc curves to the other side")
    raw = input(f"  Enter value [default {ARC_BULGE_RATIO}]: ").strip()
    if raw == "":
        return ARC_BULGE_RATIO
    try:
        return float(raw)
    except ValueError:
        print(f"  Invalid — using default {ARC_BULGE_RATIO}")
        return ARC_BULGE_RATIO


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed 4-vertex polylines found. Draw room outlines with RECTANG.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"   {x1-x0:.1f} × {y1-y0:.1f} units")

    sel = input("\nWhich rectangles to fill? (e.g. 1,2  or  'all'): ").strip().lower()
    chosen = list(range(len(rectangles))) if sel == "all" else [
        int(s.strip()) - 1 for s in sel.split(",")
        if s.strip().isdigit() and 0 <= int(s.strip()) - 1 < len(rectangles)
    ]
    if not chosen:
        print("No valid selection."); sys.exit(0)

    fixture_name, block_name = pick_fixture()
    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found. Insert it manually once first.")
        sys.exit(1)

    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0: break
        except ValueError:
            pass
        print("  Enter a positive integer.")

    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()
    draw_wire  = input("\nDraw wire between lights? (y/n) [y]: ").strip().lower() != "n"
    bulge      = ask_bulge() if draw_wire else 0.0
    draw_label = input("Add circuit labels?         (y/n) [y]: ").strip().lower() != "n"

    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)
    ensure_layer(doc, LABEL_LAYER, color=7)

    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows, cell_w, cell_h = grid_points(x0, y0, x1, y1, n_lights)

        rect_min = min(x1 - x0, y1 - y0)
        text_h   = rect_min * LABEL_HEIGHT_RATIO
        label_dy = rect_min * LABEL_OFFSET_RATIO + text_h

        if auto_inc and reset_per_rect:
            label_index = label_start

        # Blocks
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(APoint(px, py, 0), block_name,
                                 BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
            all_refs.append(ref)
            total += 1

        # Arcs — start/end exactly on each light center
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            arcs = draw_wire_arcs(ms, pts, bulge)
            all_refs.extend(arcs)
            print(f"    Drew {len(arcs)} arc(s)  bulge={bulge:.3f}"
                  f"  grid spacing: {cell_w:.2f} × {cell_h:.2f}")

        # Labels
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                txt = f"{label_prefix}.{label_index}" if auto_inc else label_prefix
                if auto_inc:
                    label_index += 1
                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                if lbl:
                    all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'arc' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg

Found 6 rectangle(s):
  [1] (21323.9, 10669.2) → (38908.6, 16890.0)   17584.7 × 6220.8 units
  [2] (21323.9, 23021.9) → (38908.6, 29242.7)   17584.7 × 6220.8 units
  [3] (21323.9, 41728.0) → (38908.6, 47948.8)   17584.7 × 6220.8 units
  [4] (21323.9, 55348.6) → (38908.6, 61569.4)   17584.7 × 6220.8 units
  [5] (21323.9, 67137.7) → (38908.6, 73358.5)   17584.7 × 6220.8 units
  [6] (21323.9, 79316.3) → (38908.6, 85537.1)   17584.7 × 6220.8 units

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label ─────────────────────────────────────────
  Auto-increment:          E20 1          → E20.1, E20.2 …
  Auto-increment + reset:  E20 1 reset    → resets per room
  Fixed label:             E20.2          → same on all lights

── Arc curvature (bulge ratio) ──────────────────────

In [8]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline) in the drawing
2. Places chosen light fixtures in an evenly spaced grid
3. Connects lights with arc-segments using LWPolyline bulge:
   - Each segment between two lights is a single arc
   - Bulge factor directly controls curvature (EXACTLY like AutoCAD arc-in-polyline)
   - Start/end points are GUARANTEED to be the light centers
4. Adds circuit labels below each light

Bulge formula (AutoCAD standard):
   bulge = tan(included_angle / 4)
   0.0  = straight line
   0.5  = gentle arc  (≈ 53° arc)
   1.0  = semicircle  (180°)
  -0.5  = arc curves to the other side

Requirements:
    pip install pyautocad pywin32
"""

import math
import sys
from pyautocad import Autocad, APoint
import win32com.client

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"

RECTANGLE_LAYER    = None
BLOCK_SCALE        = 1.0
MARGIN_RATIO       = 0.001
LABEL_HEIGHT_RATIO = 0.04
LABEL_OFFSET_RATIO = 0.01

# Default bulge: 0.5 = gentle arc matching your screenshot style
DEFAULT_BULGE = 0.5


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts_in_snake_order, cols, rows, cell_w, cell_h).
    Snake: row 0 left→right, row 1 right→left, etc.
    """
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows, cell_w, cell_h


# ─────────────────────────────────────────────
# WIRE: LWPolyline with per-segment bulge
# ─────────────────────────────────────────────

def draw_bulge_wire(ms, pts, bulge):
    """
    Draw a single open LWPolyline through all light centers.
    Each segment carries a bulge factor so it renders as an arc.

    AutoCAD LWPolyline bulge convention
    ------------------------------------
    - Bulge is stored on the START vertex of each segment
    - Positive bulge → arc curves to the LEFT of travel direction
    - Negative bulge → arc curves to the RIGHT
    - |bulge| = tan(θ/4) where θ is the included angle of the arc
      e.g. bulge=1.0 → semicircle, bulge=0.5 → ~106° arc

    We alternate the sign row by row so every arc curves
    consistently toward the interior of the room (away from edges),
    exactly like in your reference image.

    Coordinate array layout (flat, 2D):
        [x0, y0, x1, y1, x2, y2, ...]
    Bulge array layout (one value per vertex):
        [b0, b1, b2, ..., 0.0]   ← last vertex bulge always 0
    """
    if len(pts) < 2:
        return None

    # Flat coordinate array
    flat_pts = []
    for (x, y) in pts:
        flat_pts.extend([x, y])

    # Build bulge array — one per vertex, last = 0
    # We determine the "natural" bulge sign per segment based on
    # the direction of travel so arcs always bow outward sensibly.
    bulge_vals = []
    for i in range(len(pts) - 1):
        x1, y1 = pts[i]
        x2, y2 = pts[i + 1]
        dx = x2 - x1
        dy = y2 - y1

        # If moving mostly horizontally → arc bows upward (positive bulge)
        # If moving mostly vertically   → arc bows rightward (positive bulge)
        # Alternate sign each segment to create the wave pattern
        sign = 1 if i % 2 == 0 else -1
        bulge_vals.append(sign * abs(bulge))

    bulge_vals.append(0.0)  # last vertex

    # Create the LWPolyline via COM
    try:
        pt_variant = win32com.client.VARIANT(
            win32com.client.pythoncom.VT_ARRAY | win32com.client.pythoncom.VT_R8,
            flat_pts
        )
        pline = ms.AddLightWeightPolyline(pt_variant)
        pline.Closed = False

        # Apply bulge to each vertex
        for i, b in enumerate(bulge_vals):
            pline.SetBulge(i, b)

        pline.Update()
        return pline

    except Exception as e:
        print(f"    Warning: wire error — {e}")
        return None


# ─────────────────────────────────────────────
# LABEL
# ─────────────────────────────────────────────

def add_label(ms, x, y, text, height, offset_y):
    try:
        lbl = ms.AddMText(APoint(x, y - offset_y, 0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5
        lbl.Color           = 7
        return lbl
    except Exception as e:
        print(f"    Warning: label error — {e}")
        return None


# ─────────────────────────────────────────────
# USER INPUT
# ─────────────────────────────────────────────

def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    print("\n── Circuit label ─────────────────────────────────────────")
    print("  Auto-increment:          E20 1          → E20.1, E20.2 …")
    print("  Auto-increment + reset:  E20 1 reset    → resets per room")
    print("  Fixed label:             E20.2          → same on all lights")
    raw   = input("  Your choice: ").strip()
    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start = int(parts[1])
            reset = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset
        except ValueError:
            pass
    return raw, None, False, False


def ask_bulge():
    print(f"\n── Arc curvature (bulge) ─────────────────────────────────")
    print(f"  Uses AutoCAD standard bulge: tan(arc_angle / 4)")
    print(f"  0.0  = straight line")
    print(f"  0.5  = gentle arc  (default, ≈ your screenshot)")
    print(f"  1.0  = semicircle between each pair of lights")
    print(f"  Negative values flip the arc direction")
    raw = input(f"  Enter value [default {DEFAULT_BULGE}]: ").strip()
    if raw == "":
        return DEFAULT_BULGE
    try:
        return float(raw)
    except ValueError:
        print(f"  Invalid — using default {DEFAULT_BULGE}")
        return DEFAULT_BULGE


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed 4-vertex polylines found. Draw room outlines with RECTANG.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"   {x1-x0:.1f} × {y1-y0:.1f} units")

    sel = input("\nWhich rectangles to fill? (e.g. 1,2  or  'all'): ").strip().lower()
    chosen = list(range(len(rectangles))) if sel == "all" else [
        int(s.strip()) - 1 for s in sel.split(",")
        if s.strip().isdigit() and 0 <= int(s.strip()) - 1 < len(rectangles)
    ]
    if not chosen:
        print("No valid selection."); sys.exit(0)

    fixture_name, block_name = pick_fixture()
    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found. Insert it manually once first.")
        sys.exit(1)

    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0: break
        except ValueError:
            pass
        print("  Enter a positive integer.")

    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()
    draw_wire  = input("\nDraw wire between lights? (y/n) [y]: ").strip().lower() != "n"
    bulge      = ask_bulge() if draw_wire else 0.0
    draw_label = input("Add circuit labels?         (y/n) [y]: ").strip().lower() != "n"

    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)
    ensure_layer(doc, LABEL_LAYER, color=7)

    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows, cell_w, cell_h = grid_points(x0, y0, x1, y1, n_lights)

        rect_min = min(x1 - x0, y1 - y0)
        text_h   = rect_min * LABEL_HEIGHT_RATIO
        label_dy = rect_min * LABEL_OFFSET_RATIO + text_h

        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Blocks ───────────────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(APoint(px, py, 0), block_name,
                                 BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
            all_refs.append(ref)
            total += 1

        # ── Wire (bulge polyline) ─────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = draw_bulge_wire(ms, pts, bulge)
            if wire:
                all_refs.append(wire)
                print(f"    Wire: {len(pts)-1} arc-segment(s)  bulge={bulge:.2f}"
                      f"  ≈ {math.degrees(4*math.atan(abs(bulge))):.0f}° arc angle")

        # ── Labels ────────────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                txt = f"{label_prefix}.{label_index}" if auto_inc else label_prefix
                if auto_inc:
                    label_index += 1
                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                if lbl:
                    all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'bulge-arc' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg

Found 1 rectangle(s):
  [1] (2463.7, 8561.2) → (35481.0, 30014.9)   33017.2 × 21453.7 units

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label ─────────────────────────────────────────
  Auto-increment:          E20 1          → E20.1, E20.2 …
  Auto-increment + reset:  E20 1 reset    → resets per room
  Fixed label:             E20.2          → same on all lights

── Arc curvature (bulge) ─────────────────────────────────
  Uses AutoCAD standard bulge: tan(arc_angle / 4)
  0.0  = straight line
  0.5  = gentle arc  (default, ≈ your screenshot)
  1.0  = semicircle between each pair of lights
  Negative values flip the arc direction
  Rectangle [1]: 8 lights  grid 4×2  wire=bulge-arc  labels=yes

✓ Done. 8 light(s) placed and saved.


In [11]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline) in the drawing
2. Places chosen light fixtures in an evenly spaced grid
3. Connects lights with arc-segments using LWPolyline bulge:
   - Each segment between two lights is a single arc
   - Bulge factor directly controls curvature (EXACTLY like AutoCAD arc-in-polyline)
   - Start/end points are GUARANTEED to be the light centers
4. Adds circuit labels below each light

Bulge formula (AutoCAD standard):
   bulge = tan(included_angle / 4)
   0.0  = straight line
   0.5  = gentle arc  (≈ 53° arc)
   1.0  = semicircle  (180°)
  -0.5  = arc curves to the other side

Requirements:
    pip install pyautocad pywin32
"""

import math
import sys
import array as array_mod
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"

RECTANGLE_LAYER    = None
BLOCK_SCALE        = 1.0
MARGIN_RATIO       = 0.001
LABEL_HEIGHT_RATIO = 0.04
LABEL_OFFSET_RATIO = 0.01

# Default bulge: 0.5 = gentle arc matching your screenshot style
DEFAULT_BULGE = 0.5


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts_in_snake_order, cols, rows, cell_w, cell_h).
    Snake: row 0 left→right, row 1 right→left, etc.
    """
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows, cell_w, cell_h


# ─────────────────────────────────────────────
# WIRE: LWPolyline with per-segment bulge
# ─────────────────────────────────────────────

def draw_bulge_wire(ms, pts, bulge):
    """
    Draw a single open LWPolyline through all light centers.
    Each segment carries a bulge factor so it renders as an arc.

    AutoCAD LWPolyline bulge convention
    ------------------------------------
    - Bulge is stored on the START vertex of each segment
    - Positive bulge → arc curves to the LEFT of travel direction
    - Negative bulge → arc curves to the RIGHT
    - |bulge| = tan(θ/4) where θ is the included angle of the arc
      e.g. bulge=1.0 → semicircle, bulge=0.5 → ~106° arc

    We alternate the sign row by row so every arc curves
    consistently toward the interior of the room (away from edges),
    exactly like in your reference image.

    Coordinate array layout (flat, 2D):
        [x0, y0, x1, y1, x2, y2, ...]
    Bulge array layout (one value per vertex):
        [b0, b1, b2, ..., 0.0]   ← last vertex bulge always 0
    """
    if len(pts) < 2:
        return None

    # Flat coordinate array
    flat_pts = []
    for (x, y) in pts:
        flat_pts.extend([x, y])

    # Build bulge array — one per vertex, last = 0
    # We determine the "natural" bulge sign per segment based on
    # the direction of travel so arcs always bow outward sensibly.
    bulge_vals = []
    for i in range(len(pts) - 1):
        x1, y1 = pts[i]
        x2, y2 = pts[i + 1]
        dx = x2 - x1
        dy = y2 - y1

        # If moving mostly horizontally → arc bows upward (positive bulge)
        # If moving mostly vertically   → arc bows rightward (positive bulge)
        # Alternate sign each segment to create the wave pattern
        sign = 1 if i % 2 == 0 else -1
        bulge_vals.append(sign * abs(bulge))

    bulge_vals.append(0.0)  # last vertex

    # Create the LWPolyline via COM using array.array (correct type for AutoCAD)
    try:
        pt_array = array_mod.array('d', flat_pts)
        pline = ms.AddLightWeightPolyline(pt_array)
        pline.Closed = False

        # Apply bulge to each vertex
        for i, b in enumerate(bulge_vals):
            pline.SetBulge(i, b)

        pline.Update()
        return pline

    except Exception as e:
        print(f"    Warning: wire error — {e}")
        return None


# ─────────────────────────────────────────────
# LABEL
# ─────────────────────────────────────────────

def add_label(ms, x, y, text, height, offset_y):
    try:
        lbl = ms.AddMText(APoint(x, y - offset_y, 0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5
        lbl.Color           = 7
        return lbl
    except Exception as e:
        print(f"    Warning: label error — {e}")
        return None


# ─────────────────────────────────────────────
# USER INPUT
# ─────────────────────────────────────────────

def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    print("\n── Circuit label ─────────────────────────────────────────")
    print("  Auto-increment:          E20 1          → E20.1, E20.2 …")
    print("  Auto-increment + reset:  E20 1 reset    → resets per room")
    print("  Fixed label:             E20.2          → same on all lights")
    raw   = input("  Your choice: ").strip()
    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start = int(parts[1])
            reset = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset
        except ValueError:
            pass
    return raw, None, False, False


def ask_bulge():
    print(f"\n── Arc curvature (bulge) ─────────────────────────────────")
    print(f"  Uses AutoCAD standard bulge: tan(arc_angle / 4)")
    print(f"  0.0  = straight line")
    print(f"  0.5  = gentle arc  (default, ≈ your screenshot)")
    print(f"  1.0  = semicircle between each pair of lights")
    print(f"  Negative values flip the arc direction")
    raw = input(f"  Enter value [default {DEFAULT_BULGE}]: ").strip()
    if raw == "":
        return DEFAULT_BULGE
    try:
        return float(raw)
    except ValueError:
        print(f"  Invalid — using default {DEFAULT_BULGE}")
        return DEFAULT_BULGE


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed 4-vertex polylines found. Draw room outlines with RECTANG.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"   {x1-x0:.1f} × {y1-y0:.1f} units")

    sel = input("\nWhich rectangles to fill? (e.g. 1,2  or  'all'): ").strip().lower()
    chosen = list(range(len(rectangles))) if sel == "all" else [
        int(s.strip()) - 1 for s in sel.split(",")
        if s.strip().isdigit() and 0 <= int(s.strip()) - 1 < len(rectangles)
    ]
    if not chosen:
        print("No valid selection."); sys.exit(0)

    fixture_name, block_name = pick_fixture()
    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found. Insert it manually once first.")
        sys.exit(1)

    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0: break
        except ValueError:
            pass
        print("  Enter a positive integer.")

    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()
    draw_wire  = input("\nDraw wire between lights? (y/n) [y]: ").strip().lower() != "n"
    bulge      = ask_bulge() if draw_wire else 0.0
    draw_label = input("Add circuit labels?         (y/n) [y]: ").strip().lower() != "n"

    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)
    ensure_layer(doc, LABEL_LAYER, color=7)

    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows, cell_w, cell_h = grid_points(x0, y0, x1, y1, n_lights)

        rect_min = min(x1 - x0, y1 - y0)
        text_h   = rect_min * LABEL_HEIGHT_RATIO
        label_dy = rect_min * LABEL_OFFSET_RATIO + text_h

        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Blocks ───────────────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(APoint(px, py, 0), block_name,
                                 BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
            all_refs.append(ref)
            total += 1

        # ── Wire (bulge polyline) ─────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = draw_bulge_wire(ms, pts, bulge)
            if wire:
                all_refs.append(wire)
                print(f"    Wire: {len(pts)-1} arc-segment(s)  bulge={bulge:.2f}"
                      f"  ≈ {math.degrees(4*math.atan(abs(bulge))):.0f}° arc angle")

        # ── Labels ────────────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                txt = f"{label_prefix}.{label_index}" if auto_inc else label_prefix
                if auto_inc:
                    label_index += 1
                lbl = add_label(ms, px, py, txt, text_h, label_dy)
                if lbl:
                    all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'bulge-arc' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg

Found 4 rectangle(s):
  [1] (2463.7, 8561.2) → (35481.0, 30014.9)   33017.2 × 21453.7 units
  [2] (2463.7, 32834.0) → (35481.0, 54287.8)   33017.2 × 21453.7 units
  [3] (2743.8, 60180.5) → (35597.5, 66197.4)   32853.7 × 6016.9 units
  [4] (-40756.3, 77659.8) → (239542.9, 174993.7)   280299.2 × 97333.9 units

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label ─────────────────────────────────────────
  Auto-increment:          E20 1          → E20.1, E20.2 …
  Auto-increment + reset:  E20 1 reset    → resets per room
  Fixed label:             E20.2          → same on all lights

── Arc curvature (bulge) ─────────────────────────────────
  Uses AutoCAD standard bulge: tan(arc_angle / 4)
  0.0  = straight line
  0.5  = gentle arc  (default, ≈ your screenshot)
  1.0  = s

In [19]:
"""
place_lights_auto.py
--------------------
1. Detects all closed rectangles (LWPolyline) in the drawing
2. Places chosen light fixtures in an evenly spaced grid
3. Connects lights with arc-segments using LWPolyline bulge:
   - Each segment between two lights is a single arc
   - Bulge factor directly controls curvature (EXACTLY like AutoCAD arc-in-polyline)
   - Start/end points are GUARANTEED to be the light centers
4. Adds circuit labels below each light

Bulge formula (AutoCAD standard):
   bulge = tan(included_angle / 4)
   0.0  = straight line
   0.5  = gentle arc  (≈ 53° arc)
   1.0  = semicircle  (180°)
  -0.5  = arc curves to the other side

Requirements:
    pip install pyautocad pywin32
"""

import math
import sys
import array as array_mod
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "1": ("panneau_led",      "PANNEAU_LED_60x60"),
    "2": ("spot_downlight",   "SPOT_CORELINE_DN140B"),
    "3": ("hublot_etanche",   "HUBLOT_ETANCHE_11W"),
    "4": ("applique_etanche", "APPLIQUE_ETANCHE_11W"),
}

LAYER_MAP = {
    "panneau_led":      "ECLAIRAGE-PANNEAU",
    "spot_downlight":   "ECLAIRAGE-SPOT",
    "hublot_etanche":   "ECLAIRAGE-HUBLOT",
    "applique_etanche": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"

RECTANGLE_LAYER    = None
BLOCK_SCALE        = 1.0
MARGIN_RATIO       = 0.001

# Fixed label size — overridden at runtime by user input
# Set a sensible default here (in drawing units).
# Example: if your drawing is in mm and 1:100 scale, 250 = 2.5mm on paper.
DEFAULT_LABEL_HEIGHT  = 250.0   # text height in drawing units
DEFAULT_LABEL_OFFSET  = 700   # gap between light center and top of text

# Default bulge: 0.5 = gentle arc matching your screenshot style
DEFAULT_BULGE = 0.5


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    """
    Return (pts_in_snake_order, cols, rows, cell_w, cell_h).
    Snake: row 0 left→right, row 1 right→left, etc.
    """
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows, cell_w, cell_h


# ─────────────────────────────────────────────
# WIRE: LWPolyline with per-segment bulge
# ─────────────────────────────────────────────

def draw_bulge_wire(ms, pts, bulge):
    """
    Draw a single open LWPolyline through all light centers.
    Each segment carries a bulge factor so it renders as an arc.

    AutoCAD LWPolyline bulge convention
    ------------------------------------
    - Bulge is stored on the START vertex of each segment
    - Positive bulge → arc curves to the LEFT of travel direction
    - Negative bulge → arc curves to the RIGHT
    - |bulge| = tan(θ/4) where θ is the included angle of the arc
      e.g. bulge=1.0 → semicircle, bulge=0.5 → ~106° arc

    We alternate the sign row by row so every arc curves
    consistently toward the interior of the room (away from edges),
    exactly like in your reference image.

    Coordinate array layout (flat, 2D):
        [x0, y0, x1, y1, x2, y2, ...]
    Bulge array layout (one value per vertex):
        [b0, b1, b2, ..., 0.0]   ← last vertex bulge always 0
    """
    if len(pts) < 2:
        return None

    # Flat coordinate array
    flat_pts = []
    for (x, y) in pts:
        flat_pts.extend([x, y])

    # Build bulge array — one per vertex, last = 0
    # We determine the "natural" bulge sign per segment based on
    # the direction of travel so arcs always bow outward sensibly.
    bulge_vals = []
    for i in range(len(pts) - 1):
        x1, y1 = pts[i]
        x2, y2 = pts[i + 1]
        dx = x2 - x1
        dy = y2 - y1

        # If moving mostly horizontally → arc bows upward (positive bulge)
        # If moving mostly vertically   → arc bows rightward (positive bulge)
        # Alternate sign each segment to create the wave pattern
        sign = 1 if i % 2 == 0 else -1
        bulge_vals.append(sign * abs(bulge))

    bulge_vals.append(0.0)  # last vertex

    # Create the LWPolyline via COM using array.array (correct type for AutoCAD)
    try:
        pt_array = array_mod.array('d', flat_pts)
        pline = ms.AddLightWeightPolyline(pt_array)
        pline.Closed = False

        # Apply bulge to each vertex
        for i, b in enumerate(bulge_vals):
            pline.SetBulge(i, b)

        pline.Update()
        return pline

    except Exception as e:
        print(f"    Warning: wire error — {e}")
        return None


# ─────────────────────────────────────────────
# LABEL
# ─────────────────────────────────────────────

def add_label(ms, x, y, text, height, offset_y):
    try:
        lbl = ms.AddMText(APoint(x, y - offset_y, 0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 25 # default valu is 5
        lbl.Color           = 7 # Black
        return lbl
    except Exception as e:
        print(f"    Warning: label error — {e}")
        return None


# ─────────────────────────────────────────────
# USER INPUT
# ─────────────────────────────────────────────

def pick_fixture():
    print("\nAvailable fixture types:")
    for k, (name, block) in BLOCK_NAMES.items():
        print(f"  {k}. {name}  →  block: {block}")
    while True:
        choice = input("Enter fixture number: ").strip()
        if choice in BLOCK_NAMES:
            return BLOCK_NAMES[choice]
        print("  Invalid choice, try again.")


def verify_block(doc, block_name):
    try:
        doc.Blocks.Item(block_name)
        return True
    except Exception:
        return False


def ask_circuit_label():
    print("\n── Circuit label ─────────────────────────────────────────")
    print("  Auto-increment:          E20 1          → E20.1, E20.2 …")
    print("  Auto-increment + reset:  E20 1 reset    → resets per room")
    print("  Fixed label:             E20.2          → same on all lights")
    raw   = input("  Your choice: ").strip()
    parts = raw.split()
    if len(parts) >= 2:
        prefix = parts[0]
        try:
            start = int(parts[1])
            reset = len(parts) >= 3 and parts[2].lower() == "reset"
            return prefix, start, True, reset
        except ValueError:
            pass
    return raw, None, False, False


def ask_label_size():
    print(f"\n── Label text height (fixed, in drawing units) ───────────")
    print(f"  Same height will be used for ALL rooms regardless of size.")
    print(f"  Tip: height = paper_height_mm × plot_scale")
    print(f"       e.g. 2.5mm text at 1:100 → enter 250")
    raw = input(f"  Enter height [default {DEFAULT_LABEL_HEIGHT}]: ").strip()
    if raw == "":
        height = DEFAULT_LABEL_HEIGHT
    else:
        try:
            height = float(raw)
        except ValueError:
            print(f"  Invalid — using default {DEFAULT_LABEL_HEIGHT}")
            height = DEFAULT_LABEL_HEIGHT

    raw2 = input(f"  Enter offset below light [default {DEFAULT_LABEL_OFFSET}]: ").strip()
    if raw2 == "":
        offset = DEFAULT_LABEL_OFFSET
    else:
        try:
            offset = float(raw2)
        except ValueError:
            offset = DEFAULT_LABEL_OFFSET

    return height, offset



    print(f"\n── Arc curvature (bulge) ─────────────────────────────────")
    print(f"  Uses AutoCAD standard bulge: tan(arc_angle / 4)")
    print(f"  0.0  = straight line")
    print(f"  0.5  = gentle arc  (default, ≈ your screenshot)")
    print(f"  1.0  = semicircle between each pair of lights")
    print(f"  Negative values flip the arc direction")
    raw = input(f"  Enter value [default {DEFAULT_BULGE}]: ").strip()
    if raw == "":
        return DEFAULT_BULGE
    try:
        return float(raw)
    except ValueError:
        print(f"  Invalid — using default {DEFAULT_BULGE}")
        return DEFAULT_BULGE


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    try:
        acad = Autocad(create_if_not_exists=False)
        doc  = acad.doc
        ms   = acad.model
    except Exception as e:
        print(f"ERROR: Cannot connect to AutoCAD — {e}")
        sys.exit(1)

    print(f"Connected to: {doc.Name}")

    rectangles = []
    for entity in ms:
        if RECTANGLE_LAYER and entity.Layer != RECTANGLE_LAYER:
            continue
        bounds = get_rectangle_bounds(entity)
        if bounds:
            rectangles.append((entity, bounds))

    if not rectangles:
        print("No closed 4-vertex polylines found. Draw room outlines with RECTANG.")
        sys.exit(0)

    print(f"\nFound {len(rectangles)} rectangle(s):")
    for i, (_, (x0, y0, x1, y1)) in enumerate(rectangles, 1):
        print(f"  [{i}] ({x0:.1f}, {y0:.1f}) → ({x1:.1f}, {y1:.1f})"
              f"   {x1-x0:.1f} × {y1-y0:.1f} units")

    sel = input("\nWhich rectangles to fill? (e.g. 1,2  or  'all'): ").strip().lower()
    chosen = list(range(len(rectangles))) if sel == "all" else [
        int(s.strip()) - 1 for s in sel.split(",")
        if s.strip().isdigit() and 0 <= int(s.strip()) - 1 < len(rectangles)
    ]
    if not chosen:
        print("No valid selection."); sys.exit(0)

    fixture_name, block_name = pick_fixture()
    if not verify_block(doc, block_name):
        print(f"ERROR: Block '{block_name}' not found. Insert it manually once first.")
        sys.exit(1)

    while True:
        try:
            n_lights = int(input("How many lights per rectangle? ").strip())
            if n_lights > 0: break
        except ValueError:
            pass
        print("  Enter a positive integer.")

    label_prefix, label_start, auto_inc, reset_per_rect = ask_circuit_label()
    draw_wire  = input("\nDraw wire between lights? (y/n) [y]: ").strip().lower() != "n"
    bulge      = ask_bulge() if draw_wire else 0.0
    draw_label = input("Add circuit labels?         (y/n) [y]: ").strip().lower() != "n"
    label_h, label_off = ask_label_size() if draw_label else (DEFAULT_LABEL_HEIGHT, DEFAULT_LABEL_OFFSET)

    light_layer = LAYER_MAP.get(fixture_name, "ECLAIRAGE")
    ensure_layer(doc, light_layer)
    ensure_layer(doc, WIRE_LAYER,  color=6)
    ensure_layer(doc, LABEL_LAYER, color=7)

    total       = 0
    label_index = label_start if auto_inc else None
    all_refs    = []

    for idx in chosen:
        _, (x0, y0, x1, y1) = rectangles[idx]
        pts, cols, rows, cell_w, cell_h = grid_points(x0, y0, x1, y1, n_lights)

        if auto_inc and reset_per_rect:
            label_index = label_start

        # ── Blocks ───────────────────────────────
        doc.ActiveLayer = doc.Layers.Item(light_layer)
        for (px, py) in pts:
            ref = ms.InsertBlock(APoint(px, py, 0), block_name,
                                 BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
            all_refs.append(ref)
            total += 1

        # ── Wire (bulge polyline) ─────────────────
        if draw_wire and len(pts) >= 2:
            doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
            wire = draw_bulge_wire(ms, pts, bulge)
            if wire:
                all_refs.append(wire)
                print(f"    Wire: {len(pts)-1} arc-segment(s)  bulge={bulge:.2f}"
                      f"  ≈ {math.degrees(4*math.atan(abs(bulge))):.0f}° arc angle")

        # ── Labels ────────────────────────────────
        if draw_label:
            doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
            for (px, py) in pts:
                txt = f"{label_prefix}.{label_index}" if auto_inc else label_prefix
                if auto_inc:
                    label_index += 1
                lbl = add_label(ms, px, py, txt, label_h, label_off)
                if lbl:
                    all_refs.append(lbl)

        print(f"  Rectangle [{idx+1}]: {len(pts)} lights  grid {cols}×{rows}"
              f"  wire={'bulge-arc' if draw_wire else 'no'}"
              f"  labels={'yes' if draw_label else 'no'}")

    doc.ActiveLayer = doc.Layers.Item("0")
    acad.app.ZoomExtents()
    doc.Regen(True)
    doc.Save()

    print(f"\n✓ Done. {total} light(s) placed and saved.")


if __name__ == "__main__":
    main()

Connected to: Drawing1.dwg

Found 4 rectangle(s):
  [1] (-40756.3, 77659.8) → (239542.9, 174993.7)   280299.2 × 97333.9 units
  [2] (-40756.3, 192447.5) → (239542.9, 289781.4)   280299.2 × 97333.9 units
  [3] (-40756.3, 309066.6) → (239542.9, 406400.5)   280299.2 × 97333.9 units
  [4] (-40756.3, 450925.3) → (239542.9, 548259.3)   280299.2 × 97333.9 units

Available fixture types:
  1. panneau_led  →  block: PANNEAU_LED_60x60
  2. spot_downlight  →  block: SPOT_CORELINE_DN140B
  3. hublot_etanche  →  block: HUBLOT_ETANCHE_11W
  4. applique_etanche  →  block: APPLIQUE_ETANCHE_11W

── Circuit label ─────────────────────────────────────────
  Auto-increment:          E20 1          → E20.1, E20.2 …
  Auto-increment + reset:  E20 1 reset    → resets per room
  Fixed label:             E20.2          → same on all lights

── Arc curvature (bulge) ─────────────────────────────────
  Uses AutoCAD standard bulge: tan(arc_angle / 4)
  0.0  = straight line
  0.5  = gentle arc  (default, ≈ your sc

In [22]:
"""
lighting_app.py
---------------
Tkinter mini-app for placing lights in AutoCAD.
Run while AutoCAD is open with your drawing loaded.

Requirements:
    pip install pyautocad pywin32
    (tkinter is built into Python)
"""

import math
import sys
import array as array_mod
import threading
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
from pyautocad import Autocad, APoint

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "Panneau LED 60×60":    "PANNEAU_LED_60x60",
    "Spot CoreLine DN140B": "SPOT_CORELINE_DN140B",
    "Hublot étanche 11W":   "HUBLOT_ETANCHE_11W",
    "Applique étanche 11W": "APPLIQUE_ETANCHE_11W",
}

LAYER_MAP = {
    "PANNEAU_LED_60x60":    "ECLAIRAGE-PANNEAU",
    "SPOT_CORELINE_DN140B": "ECLAIRAGE-SPOT",
    "HUBLOT_ETANCHE_11W":   "ECLAIRAGE-HUBLOT",
    "APPLIQUE_ETANCHE_11W": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"
MARGIN_RATIO = 0.001
BLOCK_SCALE  = 1.0

# ─────────────────────────────────────────────
# CORE LOGIC (same as CLI version)
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows


def draw_bulge_wire(ms, pts, bulge):
    if len(pts) < 2:
        return None
    flat_pts = []
    for (x, y) in pts:
        flat_pts.extend([x, y])
    bulge_vals = []
    for i in range(len(pts) - 1):
        sign = 1 if i % 2 == 0 else -1
        bulge_vals.append(sign * abs(bulge))
    bulge_vals.append(0.0)
    try:
        pt_array = array_mod.array('d', flat_pts)
        pline = ms.AddLightWeightPolyline(pt_array)
        pline.Closed = False
        for i, b in enumerate(bulge_vals):
            pline.SetBulge(i, b)
        pline.Update()
        return pline
    except Exception as e:
        return None


def add_label(ms, x, y, text, height, offset_y):
    try:
        lbl = ms.AddMText(APoint(x, y - offset_y, 0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5
        lbl.Color           = 7
        return lbl
    except Exception:
        return None


# ─────────────────────────────────────────────
# TKINTER APP
# ─────────────────────────────────────────────

class LightingApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("AutoCAD Lighting Placer")
        self.resizable(False, False)
        self.configure(bg="#1e1e2e")

        self.rectangles = []   # list of (entity, bounds)
        self.acad = None
        self.doc  = None
        self.ms   = None

        self._build_ui()
        self._connect_autocad()

    # ── UI BUILD ────────────────────────────

    def _build_ui(self):
        PAD  = 12
        FONT = ("Segoe UI", 10)
        FONT_BOLD = ("Segoe UI", 10, "bold")
        BG   = "#1e1e2e"
        CARD = "#2a2a3e"
        ACC  = "#7c3aed"   # purple accent
        FG   = "#e2e8f0"
        ENTRY_BG = "#313145"

        style = ttk.Style(self)
        style.theme_use("clam")
        style.configure("TLabel",      background=CARD,  foreground=FG,  font=FONT)
        style.configure("TFrame",      background=CARD)
        style.configure("TLabelframe", background=CARD,  foreground=FG,  font=FONT_BOLD)
        style.configure("TLabelframe.Label", background=CARD, foreground=ACC, font=FONT_BOLD)
        style.configure("TCombobox",   fieldbackground=ENTRY_BG, background=ENTRY_BG,
                        foreground=FG, font=FONT)
        style.configure("TCheckbutton", background=CARD, foreground=FG, font=FONT)
        style.map("TCheckbutton", background=[("active", CARD)])

        outer = tk.Frame(self, bg=BG, padx=PAD, pady=PAD)
        outer.pack(fill="both", expand=True)

        # ── Header ──────────────────────────
        hdr = tk.Frame(outer, bg=ACC, pady=8)
        hdr.pack(fill="x", pady=(0, PAD))
        tk.Label(hdr, text="⚡  AutoCAD Lighting Placer",
                 font=("Segoe UI", 14, "bold"), bg=ACC, fg="white").pack()

        # ── Status bar ──────────────────────
        self.status_var = tk.StringVar(value="Connecting to AutoCAD…")
        status_bar = tk.Frame(outer, bg=CARD, padx=8, pady=5)
        status_bar.pack(fill="x", pady=(0, PAD))
        self.status_dot = tk.Label(status_bar, text="●", font=("Segoe UI", 12),
                                   bg=CARD, fg="#f59e0b")
        self.status_dot.pack(side="left")
        tk.Label(status_bar, textvariable=self.status_var,
                 font=FONT, bg=CARD, fg=FG).pack(side="left", padx=6)

        # ── Two columns ─────────────────────
        cols = tk.Frame(outer, bg=BG)
        cols.pack(fill="both")

        left  = tk.Frame(cols, bg=BG)
        right = tk.Frame(cols, bg=BG)
        left.pack(side="left", fill="both", padx=(0, 6))
        right.pack(side="left", fill="both")

        # ── LEFT: Room + Fixture ─────────────
        room_frame = ttk.LabelFrame(left, text="  Room Selection", padding=10)
        room_frame.pack(fill="x", pady=(0, 8))

        tk.Label(room_frame, text="Rectangles found:", bg=CARD, fg=FG,
                 font=FONT).grid(row=0, column=0, sticky="w", pady=2)

        self.rect_listbox = tk.Listbox(
            room_frame, height=5, selectmode="multiple",
            bg=ENTRY_BG, fg=FG, font=FONT,
            selectbackground=ACC, selectforeground="white",
            borderwidth=0, highlightthickness=1,
            highlightcolor=ACC, relief="flat"
        )
        self.rect_listbox.grid(row=1, column=0, columnspan=2, sticky="ew", pady=4)

        btn_row = tk.Frame(room_frame, bg=CARD)
        btn_row.grid(row=2, column=0, columnspan=2, sticky="ew")
        self._btn(btn_row, "⟳  Scan", self._scan_rectangles, ACC).pack(side="left", padx=(0,4))
        self._btn(btn_row, "Select All", self._select_all).pack(side="left")

        fix_frame = ttk.LabelFrame(left, text="  Fixture", padding=10)
        fix_frame.pack(fill="x", pady=(0, 8))

        tk.Label(fix_frame, text="Type:", bg=CARD, fg=FG, font=FONT).grid(
            row=0, column=0, sticky="w", pady=2)
        self.fixture_var = tk.StringVar()
        fix_combo = ttk.Combobox(fix_frame, textvariable=self.fixture_var,
                                 values=list(BLOCK_NAMES.keys()),
                                 state="readonly", width=26)
        fix_combo.grid(row=0, column=1, sticky="ew", padx=(6,0))
        fix_combo.current(0)

        tk.Label(fix_frame, text="Count per room:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.count_var = tk.StringVar(value="6")
        self._entry(fix_frame, self.count_var).grid(row=1, column=1, sticky="ew", padx=(6,0))

        # ── RIGHT: Wire + Label ──────────────
        wire_frame = ttk.LabelFrame(right, text="  Wire", padding=10)
        wire_frame.pack(fill="x", pady=(0, 8))

        self.wire_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(wire_frame, text="Draw arc wire", variable=self.wire_var,
                        command=self._toggle_wire).grid(row=0, column=0, columnspan=2,
                                                        sticky="w", pady=2)

        tk.Label(wire_frame, text="Bulge:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.bulge_var = tk.DoubleVar(value=0.5)
        self.bulge_scale = tk.Scale(
            wire_frame, from_=-1.0, to=1.0, resolution=0.05,
            orient="horizontal", variable=self.bulge_var,
            bg=CARD, fg=FG, troughcolor=ENTRY_BG,
            highlightthickness=0, activebackground=ACC,
            length=160, font=("Segoe UI", 8)
        )
        self.bulge_scale.grid(row=1, column=1, sticky="ew", padx=(6,0))

        self.bulge_lbl = tk.Label(wire_frame, text="≈ 106° arc",
                                  bg=CARD, fg="#94a3b8", font=("Segoe UI", 9))
        self.bulge_lbl.grid(row=2, column=1, sticky="w", padx=(6,0))
        self.bulge_var.trace_add("write", self._update_bulge_label)

        lbl_frame = ttk.LabelFrame(right, text="  Circuit Labels", padding=10)
        lbl_frame.pack(fill="x", pady=(0, 8))

        self.label_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(lbl_frame, text="Add labels", variable=self.label_var,
                        command=self._toggle_labels).grid(row=0, column=0, columnspan=2,
                                                          sticky="w", pady=2)

        tk.Label(lbl_frame, text="Prefix:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.prefix_var = tk.StringVar(value="E20")
        self._entry(lbl_frame, self.prefix_var, width=8).grid(row=1, column=1,
                                                               sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Start #:", bg=CARD, fg=FG, font=FONT).grid(
            row=2, column=0, sticky="w", pady=2)
        self.start_var = tk.StringVar(value="1")
        self._entry(lbl_frame, self.start_var, width=5).grid(row=2, column=1,
                                                              sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Text height:", bg=CARD, fg=FG, font=FONT).grid(
            row=3, column=0, sticky="w", pady=2)
        self.texth_var = tk.StringVar(value="250")
        self._entry(lbl_frame, self.texth_var, width=8).grid(row=3, column=1,
                                                              sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Offset:", bg=CARD, fg=FG, font=FONT).grid(
            row=4, column=0, sticky="w", pady=2)
        self.offset_var = tk.StringVar(value="700")
        self._entry(lbl_frame, self.offset_var, width=8).grid(row=4, column=1,
                                                               sticky="w", padx=(6,0))

        self.reset_var = tk.BooleanVar(value=False)
        ttk.Checkbutton(lbl_frame, text="Reset counter per room",
                        variable=self.reset_var).grid(row=5, column=0, columnspan=2,
                                                       sticky="w", pady=2)

        # ── Place button ────────────────────
        btn_frame = tk.Frame(outer, bg=BG, pady=8)
        btn_frame.pack(fill="x")
        self._btn(btn_frame, "⚡  Place Lights", self._run, ACC,
                  font=("Segoe UI", 11, "bold"), pady=10).pack(fill="x")

        # ── Log ─────────────────────────────
        log_frame = ttk.LabelFrame(outer, text="  Log", padding=6)
        log_frame.pack(fill="both", expand=True, pady=(8,0))
        self.log = scrolledtext.ScrolledText(
            log_frame, height=8, bg="#0f0f1a", fg="#a5f3fc",
            font=("Consolas", 9), borderwidth=0, relief="flat",
            insertbackground="white"
        )
        self.log.pack(fill="both", expand=True)
        self.log.configure(state="disabled")

    def _btn(self, parent, text, cmd, bg="#3f3f5a", font=("Segoe UI", 10), pady=6):
        return tk.Button(parent, text=text, command=cmd,
                         bg=bg, fg="white", font=font,
                         relief="flat", cursor="hand2",
                         activebackground="#5b5b7a",
                         activeforeground="white",
                         padx=12, pady=pady, bd=0)

    def _entry(self, parent, var, width=12):
        return tk.Entry(parent, textvariable=var, width=width,
                        bg="#313145", fg="#e2e8f0", font=("Segoe UI", 10),
                        relief="flat", insertbackground="white",
                        highlightthickness=1, highlightcolor="#7c3aed")

    # ── HELPERS ─────────────────────────────

    def _log(self, msg, color=None):
        self.log.configure(state="normal")
        self.log.insert("end", msg + "\n")
        self.log.see("end")
        self.log.configure(state="disabled")

    def _set_status(self, msg, ok=True):
        self.status_var.set(msg)
        self.status_dot.configure(fg="#22c55e" if ok else "#ef4444")

    def _update_bulge_label(self, *_):
        b = self.bulge_var.get()
        if abs(b) < 0.01:
            txt = "straight line"
        else:
            deg = math.degrees(4 * math.atan(abs(b)))
            txt = f"≈ {deg:.0f}° arc"
        self.bulge_lbl.configure(text=txt)

    def _toggle_wire(self):
        state = "normal" if self.wire_var.get() else "disabled"
        self.bulge_scale.configure(state=state)

    def _toggle_labels(self):
        pass  # fields stay visible; logic skips if unchecked

    def _select_all(self):
        self.rect_listbox.select_set(0, "end")

    # ── AUTOCAD CONNECTION ───────────────────

    def _connect_autocad(self):
        def _try():
            err_msg = ""
            connected = False

            # Try win32com.GetActiveObject first — works even when pyautocad fails
            try:
                import win32com.client
                acad_com = win32com.client.GetActiveObject("AutoCAD.Application")
                self.doc = acad_com.ActiveDocument
                self.ms  = self.doc.ModelSpace
                self.acad = type("_acad", (), {
                    "app": acad_com,
                    "doc": self.doc,
                    "model": self.ms
                })()
                connected = True
            except Exception as ex:
                err_msg = str(ex)

            # Fallback: pyautocad
            if not connected:
                try:
                    self.acad = Autocad(create_if_not_exists=False)
                    self.doc  = self.acad.doc
                    self.ms   = self.acad.model
                    connected = True
                except Exception as ex2:
                    err_msg = f"{err_msg} | {ex2}"

            if connected:
                name = self.doc.Name
                self.after(0, lambda: self._set_status(f"Connected: {name}", ok=True))
                self.after(0, self._scan_rectangles)
                self.after(0, lambda: self._log(f"✓ Connected to {name}"))
            else:
                msg = f"✗ Cannot connect: {err_msg}"
                self.after(0, lambda: self._set_status("Not connected to AutoCAD", ok=False))
                self.after(0, lambda: self._log(msg))

        threading.Thread(target=_try, daemon=True).start()

    # ── SCAN ────────────────────────────────

    def _scan_rectangles(self):
        if not self.ms:
            self._log("✗ Not connected to AutoCAD")
            return
        self.rectangles = []
        self.rect_listbox.delete(0, "end")
        try:
            for entity in self.ms:
                bounds = get_rectangle_bounds(entity)
                if bounds:
                    self.rectangles.append((entity, bounds))
            for i, (_, (x0, y0, x1, y1)) in enumerate(self.rectangles):
                label = f"[{i+1}]  {x1-x0:.0f} × {y1-y0:.0f}  @ ({x0:.0f}, {y0:.0f})"
                self.rect_listbox.insert("end", label)
            self._log(f"↺ Scanned: {len(self.rectangles)} rectangle(s) found")
        except Exception as e:
            self._log(f"✗ Scan error: {e}")

    # ── PLACE ────────────────────────────────

    def _run(self):
        if not self.ms:
            messagebox.showerror("Error", "Not connected to AutoCAD.")
            return

        sel = list(self.rect_listbox.curselection())
        if not sel:
            messagebox.showwarning("No selection", "Select at least one rectangle.")
            return

        fixture_label = self.fixture_var.get()
        block_name    = BLOCK_NAMES.get(fixture_label)
        if not block_name:
            messagebox.showerror("Error", "Invalid fixture type."); return

        try:
            n_lights = int(self.count_var.get())
            assert n_lights > 0
        except Exception:
            messagebox.showerror("Error", "Light count must be a positive integer."); return

        draw_wire  = self.wire_var.get()
        bulge      = self.bulge_var.get()
        draw_label = self.label_var.get()
        prefix     = self.prefix_var.get().strip()
        reset_per  = self.reset_var.get()

        try:
            label_start = int(self.start_var.get())
        except Exception:
            label_start = 1

        try:
            text_h  = float(self.texth_var.get())
            text_off= float(self.offset_var.get())
        except Exception:
            text_h, text_off = 250.0, 700.0

        # Run in background thread to keep UI responsive
        threading.Thread(
            target=self._place_worker,
            args=(sel, block_name, n_lights, draw_wire, bulge,
                  draw_label, prefix, label_start, reset_per, text_h, text_off),
            daemon=True
        ).start()

    def _place_worker(self, sel, block_name, n_lights, draw_wire, bulge,
                      draw_label, prefix, label_start, reset_per,
                      text_h, text_off):
        doc = self.doc
        ms  = self.ms
        self.after(0, lambda: self._log("─" * 48))

        # Verify block exists
        try:
            doc.Blocks.Item(block_name)
        except Exception:
            self.after(0, lambda: self._log(
                f"✗ Block '{block_name}' not found. Insert it manually first."))
            return

        light_layer = LAYER_MAP.get(block_name, "ECLAIRAGE")
        ensure_layer(doc, light_layer)
        ensure_layer(doc, WIRE_LAYER,  color=6)
        ensure_layer(doc, LABEL_LAYER, color=7)

        total       = 0
        all_refs    = []
        label_index = label_start

        for idx in sel:
            _, (x0, y0, x1, y1) = self.rectangles[idx]
            pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

            if reset_per:
                label_index = label_start

            # Blocks
            doc.ActiveLayer = doc.Layers.Item(light_layer)
            for (px, py) in pts:
                ref = ms.InsertBlock(APoint(px, py, 0), block_name,
                                     BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
                all_refs.append(ref)
                total += 1

            # Wire
            if draw_wire and len(pts) >= 2:
                doc.ActiveLayer = doc.Layers.Item(WIRE_LAYER)
                wire = draw_bulge_wire(ms, pts, bulge)
                if wire:
                    all_refs.append(wire)

            # Labels
            if draw_label:
                doc.ActiveLayer = doc.Layers.Item(LABEL_LAYER)
                for (px, py) in pts:
                    txt = f"{prefix}.{label_index}"
                    label_index += 1
                    lbl = add_label(ms, px, py, txt, text_h, text_off)
                    if lbl:
                        all_refs.append(lbl)

            msg = (f"✓ Room [{idx+1}]: {len(pts)} lights  {cols}×{rows} grid"
                   f"  wire={'arc' if draw_wire else 'no'}"
                   f"  labels={'yes' if draw_label else 'no'}")
            self.after(0, lambda m=msg: self._log(m))

        doc.ActiveLayer = doc.Layers.Item("0")
        self.doc.Application.ZoomExtents()
        doc.Regen(True)
        doc.Save()

        summary = f"✓ Done — {total} light(s) placed and saved."
        self.after(0, lambda: self._log(summary))
        self.after(0, lambda: self._set_status(summary, ok=True))
        self.after(0, lambda: messagebox.showinfo("Done", summary))


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    app = LightingApp()
    app.mainloop()

In [24]:
import win32com.client
try:
    acad = win32com.client.GetActiveObject("AutoCAD.Application")
    print("Connected:", acad.ActiveDocument.Name)
except Exception as e:
    print("Failed:", e)

    # List all running COM objects that contain "AutoCAD"
    print("\nSearching for AutoCAD COM name...")
    import subprocess
    result = subprocess.run(
        ['powershell', '-Command',
         'Get-Process | Where-Object {$_.MainWindowTitle -like "*AutoCAD*"} | Select-Object Name, Id, MainWindowTitle'],
        capture_output=True, text=True
    )
    print(result.stdout or "No AutoCAD process found in window titles")

Connected: Drawing1.dwg


In [1]:
"""
lighting_app.py
---------------
Tkinter mini-app for placing lights in AutoCAD.
Run while AutoCAD is open with your drawing loaded.

Requirements:
    pip install pyautocad pywin32
    (tkinter is built into Python)
"""

import math
import sys
import array as array_mod
import threading
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import win32com.client
import pythoncom

def make_point(x, y, z=0.0):
    """Create a VARIANT 3D point compatible with raw AutoCAD COM."""
    pt = win32com.client.VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_R8, [x, y, z])
    return pt

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "Panneau LED 60×60":    "PANNEAU_LED_60x60",
    "Spot CoreLine DN140B": "SPOT_CORELINE_DN140B",
    "Hublot étanche 11W":   "HUBLOT_ETANCHE_11W",
    "Applique étanche 11W": "APPLIQUE_ETANCHE_11W",
}

LAYER_MAP = {
    "PANNEAU_LED_60x60":    "ECLAIRAGE-PANNEAU",
    "SPOT_CORELINE_DN140B": "ECLAIRAGE-SPOT",
    "HUBLOT_ETANCHE_11W":   "ECLAIRAGE-HUBLOT",
    "APPLIQUE_ETANCHE_11W": "ECLAIRAGE-APPLIQUE",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"
MARGIN_RATIO = 0.001
BLOCK_SCALE  = 1.0

# ─────────────────────────────────────────────
# CORE LOGIC (same as CLI version)
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows


def draw_bulge_wire(ms, pts, bulge):
    if len(pts) < 2:
        return None
    flat_pts = []
    for (x, y) in pts:
        flat_pts.extend([x, y])
    bulge_vals = []
    for i in range(len(pts) - 1):
        sign = 1 if i % 2 == 0 else -1
        bulge_vals.append(sign * abs(bulge))
    bulge_vals.append(0.0)

    # AutoCAD COM requires a VARIANT array of doubles
    pt_variant = win32com.client.VARIANT(
        pythoncom.VT_ARRAY | pythoncom.VT_R8, flat_pts)

    pline = ms.AddLightWeightPolyline(pt_variant)
    pline = win32com.client.Dispatch(pline)
    pline.Closed = False
    for i, b in enumerate(bulge_vals):
        pline.SetBulge(i, b)
    pline.Update()
    return pline


def add_label(ms, x, y, text, height, offset_y):
    try:
        lbl = ms.AddMText(make_point(x, y - offset_y, 0.0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5
        lbl.Color           = 7
        return lbl
    except Exception:
        return None


# ─────────────────────────────────────────────
# TKINTER APP
# ─────────────────────────────────────────────

class LightingApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("AutoCAD Lighting Placer")
        self.resizable(False, False)
        self.configure(bg="#1e1e2e")

        self.rectangles = []   # list of bounds tuples (plain Python, thread-safe)
        self._acad = None
        self._doc  = None
        self._ms   = None
        import queue
        self._job_queue = queue.Queue()

        self._build_ui()
        self._connect_autocad()

    # ── UI BUILD ────────────────────────────

    def _build_ui(self):
        PAD  = 12
        FONT = ("Segoe UI", 10)
        FONT_BOLD = ("Segoe UI", 10, "bold")
        BG   = "#1e1e2e"
        CARD = "#2a2a3e"
        ACC  = "#7c3aed"   # purple accent
        FG   = "#e2e8f0"
        ENTRY_BG = "#313145"

        style = ttk.Style(self)
        style.theme_use("clam")
        style.configure("TLabel",      background=CARD,  foreground=FG,  font=FONT)
        style.configure("TFrame",      background=CARD)
        style.configure("TLabelframe", background=CARD,  foreground=FG,  font=FONT_BOLD)
        style.configure("TLabelframe.Label", background=CARD, foreground=ACC, font=FONT_BOLD)
        style.configure("TCombobox",   fieldbackground=ENTRY_BG, background=ENTRY_BG,
                        foreground=FG, font=FONT)
        style.configure("TCheckbutton", background=CARD, foreground=FG, font=FONT)
        style.map("TCheckbutton", background=[("active", CARD)])

        outer = tk.Frame(self, bg=BG, padx=PAD, pady=PAD)
        outer.pack(fill="both", expand=True)

        # ── Header ──────────────────────────
        hdr = tk.Frame(outer, bg=ACC, pady=8)
        hdr.pack(fill="x", pady=(0, PAD))
        tk.Label(hdr, text="⚡  AutoCAD Lighting Placer",
                 font=("Segoe UI", 14, "bold"), bg=ACC, fg="white").pack()

        # ── Status bar ──────────────────────
        self.status_var = tk.StringVar(value="Connecting to AutoCAD…")
        status_bar = tk.Frame(outer, bg=CARD, padx=8, pady=5)
        status_bar.pack(fill="x", pady=(0, PAD))
        self.status_dot = tk.Label(status_bar, text="●", font=("Segoe UI", 12),
                                   bg=CARD, fg="#f59e0b")
        self.status_dot.pack(side="left")
        tk.Label(status_bar, textvariable=self.status_var,
                 font=FONT, bg=CARD, fg=FG).pack(side="left", padx=6)

        # ── Two columns ─────────────────────
        cols = tk.Frame(outer, bg=BG)
        cols.pack(fill="both")

        left  = tk.Frame(cols, bg=BG)
        right = tk.Frame(cols, bg=BG)
        left.pack(side="left", fill="both", padx=(0, 6))
        right.pack(side="left", fill="both")

        # ── LEFT: Room + Fixture ─────────────
        room_frame = ttk.LabelFrame(left, text="  Room Selection", padding=10)
        room_frame.pack(fill="x", pady=(0, 8))

        tk.Label(room_frame, text="Rectangles found:", bg=CARD, fg=FG,
                 font=FONT).grid(row=0, column=0, sticky="w", pady=2)

        self.rect_listbox = tk.Listbox(
            room_frame, height=5, selectmode="multiple",
            bg=ENTRY_BG, fg=FG, font=FONT,
            selectbackground=ACC, selectforeground="white",
            borderwidth=0, highlightthickness=1,
            highlightcolor=ACC, relief="flat"
        )
        self.rect_listbox.grid(row=1, column=0, columnspan=2, sticky="ew", pady=4)

        btn_row = tk.Frame(room_frame, bg=CARD)
        btn_row.grid(row=2, column=0, columnspan=2, sticky="ew")
        self._btn(btn_row, "⟳  Scan", self._scan_rectangles, ACC).pack(side="left", padx=(0,4))
        self._btn(btn_row, "Select All", self._select_all).pack(side="left", padx=(4,0))
        self._btn(btn_row, "⚡ Reconnect", self._connect_autocad, "#b45309").pack(side="left", padx=(4,0))

        fix_frame = ttk.LabelFrame(left, text="  Fixture", padding=10)
        fix_frame.pack(fill="x", pady=(0, 8))

        tk.Label(fix_frame, text="Type:", bg=CARD, fg=FG, font=FONT).grid(
            row=0, column=0, sticky="w", pady=2)
        self.fixture_var = tk.StringVar()
        fix_combo = ttk.Combobox(fix_frame, textvariable=self.fixture_var,
                                 values=list(BLOCK_NAMES.keys()),
                                 state="readonly", width=26)
        fix_combo.grid(row=0, column=1, sticky="ew", padx=(6,0))
        fix_combo.current(0)

        tk.Label(fix_frame, text="Count per room:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.count_var = tk.StringVar(value="6")
        self._entry(fix_frame, self.count_var).grid(row=1, column=1, sticky="ew", padx=(6,0))

        # ── RIGHT: Wire + Label ──────────────
        wire_frame = ttk.LabelFrame(right, text="  Wire", padding=10)
        wire_frame.pack(fill="x", pady=(0, 8))

        self.wire_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(wire_frame, text="Draw arc wire", variable=self.wire_var,
                        command=self._toggle_wire).grid(row=0, column=0, columnspan=2,
                                                        sticky="w", pady=2)

        tk.Label(wire_frame, text="Bulge:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.bulge_var = tk.DoubleVar(value=0.5)
        self.bulge_scale = tk.Scale(
            wire_frame, from_=-1.0, to=1.0, resolution=0.05,
            orient="horizontal", variable=self.bulge_var,
            bg=CARD, fg=FG, troughcolor=ENTRY_BG,
            highlightthickness=0, activebackground=ACC,
            length=160, font=("Segoe UI", 8)
        )
        self.bulge_scale.grid(row=1, column=1, sticky="ew", padx=(6,0))

        self.bulge_lbl = tk.Label(wire_frame, text="≈ 106° arc",
                                  bg=CARD, fg="#94a3b8", font=("Segoe UI", 9))
        self.bulge_lbl.grid(row=2, column=1, sticky="w", padx=(6,0))
        self.bulge_var.trace_add("write", self._update_bulge_label)

        lbl_frame = ttk.LabelFrame(right, text="  Circuit Labels", padding=10)
        lbl_frame.pack(fill="x", pady=(0, 8))

        self.label_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(lbl_frame, text="Add labels", variable=self.label_var,
                        command=self._toggle_labels).grid(row=0, column=0, columnspan=2,
                                                          sticky="w", pady=2)

        tk.Label(lbl_frame, text="Prefix:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.prefix_var = tk.StringVar(value="E20")
        self._entry(lbl_frame, self.prefix_var, width=8).grid(row=1, column=1,
                                                               sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Start #:", bg=CARD, fg=FG, font=FONT).grid(
            row=2, column=0, sticky="w", pady=2)
        self.start_var = tk.StringVar(value="1")
        self._entry(lbl_frame, self.start_var, width=5).grid(row=2, column=1,
                                                              sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Text height:", bg=CARD, fg=FG, font=FONT).grid(
            row=3, column=0, sticky="w", pady=2)
        self.texth_var = tk.StringVar(value="250")
        self._entry(lbl_frame, self.texth_var, width=8).grid(row=3, column=1,
                                                              sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Offset:", bg=CARD, fg=FG, font=FONT).grid(
            row=4, column=0, sticky="w", pady=2)
        self.offset_var = tk.StringVar(value="700")
        self._entry(lbl_frame, self.offset_var, width=8).grid(row=4, column=1,
                                                               sticky="w", padx=(6,0))

        self.reset_var = tk.BooleanVar(value=False)
        ttk.Checkbutton(lbl_frame, text="Reset counter per room",
                        variable=self.reset_var).grid(row=5, column=0, columnspan=2,
                                                       sticky="w", pady=2)

        # ── Place button ────────────────────
        btn_frame = tk.Frame(outer, bg=BG, pady=8)
        btn_frame.pack(fill="x")
        self._btn(btn_frame, "⚡  Place Lights", self._run, ACC,
                  font=("Segoe UI", 11, "bold"), pady=10).pack(fill="x")

        # ── Log ─────────────────────────────
        log_frame = ttk.LabelFrame(outer, text="  Log", padding=6)
        log_frame.pack(fill="both", expand=True, pady=(8,0))
        self.log = scrolledtext.ScrolledText(
            log_frame, height=8, bg="#0f0f1a", fg="#a5f3fc",
            font=("Consolas", 9), borderwidth=0, relief="flat",
            insertbackground="white"
        )
        self.log.pack(fill="both", expand=True)
        self.log.configure(state="disabled")

    def _btn(self, parent, text, cmd, bg="#3f3f5a", font=("Segoe UI", 10), pady=6):
        return tk.Button(parent, text=text, command=cmd,
                         bg=bg, fg="white", font=font,
                         relief="flat", cursor="hand2",
                         activebackground="#5b5b7a",
                         activeforeground="white",
                         padx=12, pady=pady, bd=0)

    def _entry(self, parent, var, width=12):
        return tk.Entry(parent, textvariable=var, width=width,
                        bg="#313145", fg="#e2e8f0", font=("Segoe UI", 10),
                        relief="flat", insertbackground="white",
                        highlightthickness=1, highlightcolor="#7c3aed")

    # ── HELPERS ─────────────────────────────

    def _log(self, msg, color=None):
        self.log.configure(state="normal")
        self.log.insert("end", msg + "\n")
        self.log.see("end")
        self.log.configure(state="disabled")

    def _set_status(self, msg, ok=True):
        self.status_var.set(msg)
        self.status_dot.configure(fg="#22c55e" if ok else "#ef4444")

    def _update_bulge_label(self, *_):
        b = self.bulge_var.get()
        if abs(b) < 0.01:
            txt = "straight line"
        else:
            deg = math.degrees(4 * math.atan(abs(b)))
            txt = f"≈ {deg:.0f}° arc"
        self.bulge_lbl.configure(text=txt)

    def _toggle_wire(self):
        state = "normal" if self.wire_var.get() else "disabled"
        self.bulge_scale.configure(state=state)

    def _toggle_labels(self):
        pass  # fields stay visible; logic skips if unchecked

    def _select_all(self):
        self.rect_listbox.select_set(0, "end")

    # ── AUTOCAD CONNECTION ───────────────────

    def _connect_autocad(self):
        """Launch one background thread that owns ALL COM work: connect + scan."""
        threading.Thread(target=self._com_worker, daemon=True).start()

    def _com_worker(self):
        """
        Single thread that owns every COM call.
        Rule: COM objects are NEVER passed to other threads.
        All AutoCAD work (connect, scan, place) happens here.
        """
        pythoncom.CoInitialize()
        try:
            acad = win32com.client.Dispatch(
                win32com.client.GetActiveObject("AutoCAD.Application"))
            doc  = win32com.client.Dispatch(acad.ActiveDocument)
            ms   = win32com.client.Dispatch(doc.ModelSpace)
        except Exception as ex:
            msg = str(ex)
            self.after(0, lambda: self._set_status("Not connected — click Reconnect", ok=False))
            self.after(0, lambda: self._log(f"✗ Cannot connect: {msg}"))
            return

        # Store references — only used from this thread via the queue
        self._acad = acad
        self._doc  = doc
        self._ms   = ms

        name = doc.Name
        self.after(0, lambda: self._set_status(f"Connected: {name}", ok=True))
        self.after(0, lambda: self._log(f"✓ Connected to {name}"))

        # Immediately scan
        self._do_scan()

        # Event loop — wait for jobs from the UI thread
        while True:
            try:
                job = self._job_queue.get(timeout=0.2)
                if job is None:
                    break
                if job[0] == "scan":
                    self._do_scan()
                elif job[0] == "place":
                    self._do_place(*job[1:])
            except Exception:
                continue

    # ── SCAN ────────────────────────────────

    def _scan_rectangles(self):
        """Called from UI — posts a scan job to the COM thread via queue."""
        try:
            self._job_queue.put(("scan",))
        except Exception:
            self._log("✗ Not connected yet — click Reconnect")

    def _do_scan(self):
        """Runs on the COM thread."""
        self.rectangles = []
        try:
            ms    = win32com.client.Dispatch(self._doc.ModelSpace)
            count = ms.Count
            for i in range(count):
                try:
                    entity = win32com.client.Dispatch(ms.Item(i))
                    bounds = get_rectangle_bounds(entity)
                    if bounds:
                        # Store only the bounds (plain Python data, safe to share)
                        self.rectangles.append(bounds)
                except Exception:
                    continue

            rects = list(self.rectangles)
            def _update():
                self.rect_listbox.delete(0, "end")
                for i, (x0, y0, x1, y1) in enumerate(rects):
                    label = f"[{i+1}]  {x1-x0:.0f} × {y1-y0:.0f}  @ ({x0:.0f}, {y0:.0f})"
                    self.rect_listbox.insert("end", label)
                self._log(f"↺ Scanned: {len(rects)} rectangle(s) found")
            self.after(0, _update)
        except Exception as e:
            err = str(e)
            self.after(0, lambda: self._log(f"✗ Scan error: {err}"))

    # ── PLACE ────────────────────────────────

    def _run(self):
        if not hasattr(self, '_job_queue') or not hasattr(self, '_doc'):
            messagebox.showerror("Error", "Not connected to AutoCAD. Click Reconnect.")
            return

        sel = list(self.rect_listbox.curselection())
        if not sel:
            messagebox.showwarning("No selection", "Select at least one rectangle.")
            return

        fixture_label = self.fixture_var.get()
        block_name    = BLOCK_NAMES.get(fixture_label)
        if not block_name:
            messagebox.showerror("Error", "Invalid fixture type."); return

        try:
            n_lights = int(self.count_var.get())
            assert n_lights > 0
        except Exception:
            messagebox.showerror("Error", "Light count must be a positive integer."); return

        draw_wire  = self.wire_var.get()
        bulge      = self.bulge_var.get()
        draw_label = self.label_var.get()
        prefix     = self.prefix_var.get().strip()
        reset_per  = self.reset_var.get()

        try:
            label_start = int(self.start_var.get())
        except Exception:
            label_start = 1

        try:
            text_h  = float(self.texth_var.get())
            text_off= float(self.offset_var.get())
        except Exception:
            text_h, text_off = 250.0, 700.0

        # Post placement job to the COM thread
        try:
            self._job_queue.put(("place", sel, block_name, n_lights, draw_wire, bulge,
                                 draw_label, prefix, label_start, reset_per, text_h, text_off))
        except Exception as e:
            messagebox.showerror("Error", f"Could not queue job: {e}")

    def _do_place(self, sel, block_name, n_lights, draw_wire, bulge,
                  draw_label, prefix, label_start, reset_per,
                  text_h, text_off):
        # Already on the COM thread — no CoInitialize needed
        doc = self._doc
        ms  = win32com.client.Dispatch(self._doc.ModelSpace)
        self.after(0, lambda: self._log("─" * 48))

        # Verify block exists
        try:
            self._doc.Blocks.Item(block_name)
        except Exception:
            self.after(0, lambda: self._log(
                f"✗ Block '{block_name}' not found. Insert it manually first."))
            return

        light_layer = LAYER_MAP.get(block_name, "ECLAIRAGE")
        ensure_layer(self._doc, light_layer)
        ensure_layer(self._doc, WIRE_LAYER,  color=6)
        ensure_layer(self._doc, LABEL_LAYER, color=7)

        total       = 0
        all_refs    = []
        label_index = label_start

        for idx in sel:
            x0, y0, x1, y1 = self.rectangles[idx]
            pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

            if reset_per:
                label_index = label_start

            # Blocks
            self._doc.ActiveLayer = self._doc.Layers.Item(light_layer)
            for (px, py) in pts:
                ref = ms.InsertBlock(make_point(px, py, 0.0), block_name,
                                     BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
                all_refs.append(ref)
                total += 1

            # Wire
            if draw_wire and len(pts) >= 2:
                self._doc.ActiveLayer = self._doc.Layers.Item(WIRE_LAYER)
                try:
                    wire = draw_bulge_wire(ms, pts, bulge)
                    if wire:
                        all_refs.append(wire)
                except Exception as we:
                    werr = str(we)
                    self.after(0, lambda: self._log(f"    ✗ Wire error: {werr}"))

            # Labels
            if draw_label:
                self._doc.ActiveLayer = self._doc.Layers.Item(LABEL_LAYER)
                for (px, py) in pts:
                    txt = f"{prefix}.{label_index}"
                    label_index += 1
                    lbl = add_label(ms, px, py, txt, text_h, text_off)
                    if lbl:
                        all_refs.append(lbl)

            msg = (f"✓ Room [{idx+1}]: {len(pts)} lights  {cols}×{rows} grid"
                   f"  wire={'arc' if draw_wire else 'no'}"
                   f"  labels={'yes' if draw_label else 'no'}")
            self.after(0, lambda m=msg: self._log(m))

        self._doc.ActiveLayer = self._doc.Layers.Item("0")
        self._acad.ZoomExtents()
        self._doc.Regen(True)
        self._doc.Save()

        summary = f"✓ Done — {total} light(s) placed and saved."
        self.after(0, lambda: self._log(summary))
        self.after(0, lambda: self._set_status(summary, ok=True))
        self.after(0, lambda: messagebox.showinfo("Done", summary))


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    app = LightingApp()
    app.mainloop()

In [2]:
"""
lighting_app.py
---------------
Tkinter mini-app for placing lights in AutoCAD.
Run while AutoCAD is open with your drawing loaded.

Requirements:
    pip install pyautocad pywin32
    (tkinter is built into Python)
"""

import math
import sys
import array as array_mod
import threading
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import win32com.client
import pythoncom

def make_point(x, y, z=0.0):
    """Create a VARIANT 3D point compatible with raw AutoCAD COM."""
    pt = win32com.client.VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_R8, [x, y, z])
    return pt

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "Panneau LED 60×60":    "PANNEAU_LED_60x60",
    "Spot CoreLine DN140B": "SPOT_CORELINE_DN140B",
    "Hublot étanche 11W":   "HUBLOT_ETANCHE_11W",
    "Applique étanche 11W": "APPLIQUE_ETANCHE_11W",
    "Brasseur d'air 75W":   "BRASSEUR_AIR_75W",
}

LAYER_MAP = {
    "PANNEAU_LED_60x60":    "ECLAIRAGE-PANNEAU",
    "SPOT_CORELINE_DN140B": "ECLAIRAGE-SPOT",
    "HUBLOT_ETANCHE_11W":   "ECLAIRAGE-HUBLOT",
    "APPLIQUE_ETANCHE_11W": "ECLAIRAGE-APPLIQUE",
    "BRASSEUR_AIR_75W":     "ECLAIRAGE-VENTILATEUR",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"
MARGIN_RATIO = 0.001
BLOCK_SCALE  = 1.0

# ─────────────────────────────────────────────
# CORE LOGIC (same as CLI version)
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows


def draw_bulge_wire(ms, pts, bulge):
    if len(pts) < 2:
        return None
    flat_pts = []
    for (x, y) in pts:
        flat_pts.extend([x, y])
    bulge_vals = []
    for i in range(len(pts) - 1):
        sign = 1 if i % 2 == 0 else -1
        bulge_vals.append(sign * abs(bulge))
    bulge_vals.append(0.0)

    # AutoCAD COM requires a VARIANT array of doubles
    pt_variant = win32com.client.VARIANT(
        pythoncom.VT_ARRAY | pythoncom.VT_R8, flat_pts)

    pline = ms.AddLightWeightPolyline(pt_variant)
    pline = win32com.client.Dispatch(pline)
    pline.Closed = False
    for i, b in enumerate(bulge_vals):
        pline.SetBulge(i, b)
    pline.Update()
    return pline


def add_label(ms, x, y, text, height, offset_y):
    try:
        lbl = ms.AddMText(make_point(x, y - offset_y, 0.0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5
        lbl.Color           = 7
        return lbl
    except Exception:
        return None


# ─────────────────────────────────────────────
# TKINTER APP
# ─────────────────────────────────────────────

class LightingApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("AutoCAD Lighting Placer")
        self.resizable(False, False)
        self.configure(bg="#1e1e2e")

        self.rectangles = []   # list of bounds tuples (plain Python, thread-safe)
        self._acad = None
        self._doc  = None
        self._ms   = None
        import queue
        self._job_queue = queue.Queue()

        self._build_ui()
        self._connect_autocad()

    # ── UI BUILD ────────────────────────────

    def _build_ui(self):
        PAD  = 12
        FONT = ("Segoe UI", 10)
        FONT_BOLD = ("Segoe UI", 10, "bold")
        BG   = "#1e1e2e"
        CARD = "#2a2a3e"
        ACC  = "#7c3aed"   # purple accent
        FG   = "#e2e8f0"
        ENTRY_BG = "#313145"

        style = ttk.Style(self)
        style.theme_use("clam")
        style.configure("TLabel",      background=CARD,  foreground=FG,  font=FONT)
        style.configure("TFrame",      background=CARD)
        style.configure("TLabelframe", background=CARD,  foreground=FG,  font=FONT_BOLD)
        style.configure("TLabelframe.Label", background=CARD, foreground=ACC, font=FONT_BOLD)
        style.configure("TCombobox",   fieldbackground=ENTRY_BG, background=ENTRY_BG,
                        foreground=FG, font=FONT)
        style.configure("TCheckbutton", background=CARD, foreground=FG, font=FONT)
        style.map("TCheckbutton", background=[("active", CARD)])

        outer = tk.Frame(self, bg=BG, padx=PAD, pady=PAD)
        outer.pack(fill="both", expand=True)

        # ── Header ──────────────────────────
        hdr = tk.Frame(outer, bg=ACC, pady=8)
        hdr.pack(fill="x", pady=(0, PAD))
        tk.Label(hdr, text="⚡  AutoCAD Lighting Placer",
                 font=("Segoe UI", 14, "bold"), bg=ACC, fg="white").pack()

        # ── Status bar ──────────────────────
        self.status_var = tk.StringVar(value="Connecting to AutoCAD…")
        status_bar = tk.Frame(outer, bg=CARD, padx=8, pady=5)
        status_bar.pack(fill="x", pady=(0, PAD))
        self.status_dot = tk.Label(status_bar, text="●", font=("Segoe UI", 12),
                                   bg=CARD, fg="#f59e0b")
        self.status_dot.pack(side="left")
        tk.Label(status_bar, textvariable=self.status_var,
                 font=FONT, bg=CARD, fg=FG).pack(side="left", padx=6)

        # ── Two columns ─────────────────────
        cols = tk.Frame(outer, bg=BG)
        cols.pack(fill="both")

        left  = tk.Frame(cols, bg=BG)
        right = tk.Frame(cols, bg=BG)
        left.pack(side="left", fill="both", padx=(0, 6))
        right.pack(side="left", fill="both")

        # ── LEFT: Room + Fixture ─────────────
        room_frame = ttk.LabelFrame(left, text="  Room Selection", padding=10)
        room_frame.pack(fill="x", pady=(0, 8))

        tk.Label(room_frame, text="Rectangles found:", bg=CARD, fg=FG,
                 font=FONT).grid(row=0, column=0, sticky="w", pady=2)

        self.rect_listbox = tk.Listbox(
            room_frame, height=5, selectmode="multiple",
            bg=ENTRY_BG, fg=FG, font=FONT,
            selectbackground=ACC, selectforeground="white",
            borderwidth=0, highlightthickness=1,
            highlightcolor=ACC, relief="flat"
        )
        self.rect_listbox.grid(row=1, column=0, columnspan=2, sticky="ew", pady=4)

        btn_row = tk.Frame(room_frame, bg=CARD)
        btn_row.grid(row=2, column=0, columnspan=2, sticky="ew")
        self._btn(btn_row, "⟳  Scan", self._scan_rectangles, ACC).pack(side="left", padx=(0,4))
        self._btn(btn_row, "Select All", self._select_all).pack(side="left", padx=(4,0))
        self._btn(btn_row, "⚡ Reconnect", self._connect_autocad, "#b45309").pack(side="left", padx=(4,0))

        fix_frame = ttk.LabelFrame(left, text="  Fixture", padding=10)
        fix_frame.pack(fill="x", pady=(0, 8))

        tk.Label(fix_frame, text="Type:", bg=CARD, fg=FG, font=FONT).grid(
            row=0, column=0, sticky="w", pady=2)
        self.fixture_var = tk.StringVar()
        fix_combo = ttk.Combobox(fix_frame, textvariable=self.fixture_var,
                                 values=list(BLOCK_NAMES.keys()),
                                 state="readonly", width=26)
        fix_combo.grid(row=0, column=1, sticky="ew", padx=(6,0))
        fix_combo.current(0)

        tk.Label(fix_frame, text="Count per room:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.count_var = tk.StringVar(value="6")
        self._entry(fix_frame, self.count_var).grid(row=1, column=1, sticky="ew", padx=(6,0))

        # ── RIGHT: Wire + Label ──────────────
        wire_frame = ttk.LabelFrame(right, text="  Wire", padding=10)
        wire_frame.pack(fill="x", pady=(0, 8))

        self.wire_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(wire_frame, text="Draw arc wire", variable=self.wire_var,
                        command=self._toggle_wire).grid(row=0, column=0, columnspan=2,
                                                        sticky="w", pady=2)

        tk.Label(wire_frame, text="Bulge:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.bulge_var = tk.DoubleVar(value=0.5)
        self.bulge_scale = tk.Scale(
            wire_frame, from_=-1.0, to=1.0, resolution=0.05,
            orient="horizontal", variable=self.bulge_var,
            bg=CARD, fg=FG, troughcolor=ENTRY_BG,
            highlightthickness=0, activebackground=ACC,
            length=160, font=("Segoe UI", 8)
        )
        self.bulge_scale.grid(row=1, column=1, sticky="ew", padx=(6,0))

        self.bulge_lbl = tk.Label(wire_frame, text="≈ 106° arc",
                                  bg=CARD, fg="#94a3b8", font=("Segoe UI", 9))
        self.bulge_lbl.grid(row=2, column=1, sticky="w", padx=(6,0))
        self.bulge_var.trace_add("write", self._update_bulge_label)

        lbl_frame = ttk.LabelFrame(right, text="  Circuit Labels", padding=10)
        lbl_frame.pack(fill="x", pady=(0, 8))

        self.label_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(lbl_frame, text="Add labels", variable=self.label_var,
                        command=self._toggle_labels).grid(row=0, column=0, columnspan=2,
                                                          sticky="w", pady=2)

        # Label mode: Fixed or Auto-increment
        tk.Label(lbl_frame, text="Mode:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.label_mode_var = tk.StringVar(value="auto")
        mode_frame = tk.Frame(lbl_frame, bg=CARD)
        mode_frame.grid(row=1, column=1, sticky="w", padx=(6,0))
        ttk.Radiobutton(mode_frame, text="Auto", variable=self.label_mode_var,
                        value="auto", command=self._toggle_label_mode).pack(side="left")
        ttk.Radiobutton(mode_frame, text="Fixed", variable=self.label_mode_var,
                        value="fixed", command=self._toggle_label_mode).pack(side="left", padx=(8,0))

        tk.Label(lbl_frame, text="Prefix:", bg=CARD, fg=FG, font=FONT).grid(
            row=2, column=0, sticky="w", pady=2)
        self.prefix_var = tk.StringVar(value="E20")
        self._entry(lbl_frame, self.prefix_var, width=8).grid(row=2, column=1,
                                                               sticky="w", padx=(6,0))

        # Auto-increment fields
        self.start_lbl = tk.Label(lbl_frame, text="Start #:", bg=CARD, fg=FG, font=FONT)
        self.start_lbl.grid(row=3, column=0, sticky="w", pady=2)
        self.start_var = tk.StringVar(value="1")
        self.start_entry = self._entry(lbl_frame, self.start_var, width=5)
        self.start_entry.grid(row=3, column=1, sticky="w", padx=(6,0))

        self.reset_var = tk.BooleanVar(value=False)
        self.reset_chk = ttk.Checkbutton(lbl_frame, text="Reset counter per room",
                                          variable=self.reset_var)
        self.reset_chk.grid(row=4, column=0, columnspan=2, sticky="w", pady=2)

        tk.Label(lbl_frame, text="Text height:", bg=CARD, fg=FG, font=FONT).grid(
            row=5, column=0, sticky="w", pady=2)
        self.texth_var = tk.StringVar(value="250")
        self._entry(lbl_frame, self.texth_var, width=8).grid(row=5, column=1,
                                                              sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Offset:", bg=CARD, fg=FG, font=FONT).grid(
            row=6, column=0, sticky="w", pady=2)
        self.offset_var = tk.StringVar(value="700")
        self._entry(lbl_frame, self.offset_var, width=8).grid(row=6, column=1,
                                                               sticky="w", padx=(6,0))

        # ── Place button ────────────────────
        btn_frame = tk.Frame(outer, bg=BG, pady=8)
        btn_frame.pack(fill="x")
        self._btn(btn_frame, "⚡  Place Lights", self._run, ACC,
                  font=("Segoe UI", 11, "bold"), pady=10).pack(fill="x")

        # ── Log ─────────────────────────────
        log_frame = ttk.LabelFrame(outer, text="  Log", padding=6)
        log_frame.pack(fill="both", expand=True, pady=(8,0))
        self.log = scrolledtext.ScrolledText(
            log_frame, height=8, bg="#0f0f1a", fg="#a5f3fc",
            font=("Consolas", 9), borderwidth=0, relief="flat",
            insertbackground="white"
        )
        self.log.pack(fill="both", expand=True)
        self.log.configure(state="disabled")

    def _btn(self, parent, text, cmd, bg="#3f3f5a", font=("Segoe UI", 10), pady=6):
        return tk.Button(parent, text=text, command=cmd,
                         bg=bg, fg="white", font=font,
                         relief="flat", cursor="hand2",
                         activebackground="#5b5b7a",
                         activeforeground="white",
                         padx=12, pady=pady, bd=0)

    def _entry(self, parent, var, width=12):
        return tk.Entry(parent, textvariable=var, width=width,
                        bg="#313145", fg="#e2e8f0", font=("Segoe UI", 10),
                        relief="flat", insertbackground="white",
                        highlightthickness=1, highlightcolor="#7c3aed")

    # ── HELPERS ─────────────────────────────

    def _log(self, msg, color=None):
        self.log.configure(state="normal")
        self.log.insert("end", msg + "\n")
        self.log.see("end")
        self.log.configure(state="disabled")

    def _set_status(self, msg, ok=True):
        self.status_var.set(msg)
        self.status_dot.configure(fg="#22c55e" if ok else "#ef4444")

    def _update_bulge_label(self, *_):
        b = self.bulge_var.get()
        if abs(b) < 0.01:
            txt = "straight line"
        else:
            deg = math.degrees(4 * math.atan(abs(b)))
            txt = f"≈ {deg:.0f}° arc"
        self.bulge_lbl.configure(text=txt)

    def _toggle_wire(self):
        state = "normal" if self.wire_var.get() else "disabled"
        self.bulge_scale.configure(state=state)

    def _toggle_labels(self):
        pass  # fields stay visible; logic skips if unchecked

    def _toggle_label_mode(self):
        """Show/hide auto-increment fields based on selected mode."""
        is_auto = self.label_mode_var.get() == "auto"
        state = "normal" if is_auto else "disabled"
        self.start_entry.configure(state=state)
        self.reset_chk.configure(state=state)
        self.start_lbl.configure(fg="#e2e8f0" if is_auto else "#555570")

    def _select_all(self):
        self.rect_listbox.select_set(0, "end")

    # ── AUTOCAD CONNECTION ───────────────────

    def _connect_autocad(self):
        """Launch one background thread that owns ALL COM work: connect + scan."""
        threading.Thread(target=self._com_worker, daemon=True).start()

    def _com_worker(self):
        """
        Single thread that owns every COM call.
        Rule: COM objects are NEVER passed to other threads.
        All AutoCAD work (connect, scan, place) happens here.
        """
        pythoncom.CoInitialize()
        try:
            acad = win32com.client.Dispatch(
                win32com.client.GetActiveObject("AutoCAD.Application"))
            doc  = win32com.client.Dispatch(acad.ActiveDocument)
            ms   = win32com.client.Dispatch(doc.ModelSpace)
        except Exception as ex:
            msg = str(ex)
            self.after(0, lambda: self._set_status("Not connected — click Reconnect", ok=False))
            self.after(0, lambda: self._log(f"✗ Cannot connect: {msg}"))
            return

        # Store references — only used from this thread via the queue
        self._acad = acad
        self._doc  = doc
        self._ms   = ms

        name = doc.Name
        self.after(0, lambda: self._set_status(f"Connected: {name}", ok=True))
        self.after(0, lambda: self._log(f"✓ Connected to {name}"))

        # Immediately scan
        self._do_scan()

        # Event loop — wait for jobs from the UI thread
        while True:
            try:
                job = self._job_queue.get(timeout=0.2)
                if job is None:
                    break
                if job[0] == "scan":
                    self._do_scan()
                elif job[0] == "place":
                    self._do_place(*job[1:])
            except Exception:
                continue

    # ── SCAN ────────────────────────────────

    def _scan_rectangles(self):
        """Called from UI — posts a scan job to the COM thread via queue."""
        try:
            self._job_queue.put(("scan",))
        except Exception:
            self._log("✗ Not connected yet — click Reconnect")

    def _do_scan(self):
        """Runs on the COM thread."""
        self.rectangles = []
        try:
            ms    = win32com.client.Dispatch(self._doc.ModelSpace)
            count = ms.Count
            for i in range(count):
                try:
                    entity = win32com.client.Dispatch(ms.Item(i))
                    bounds = get_rectangle_bounds(entity)
                    if bounds:
                        # Store only the bounds (plain Python data, safe to share)
                        self.rectangles.append(bounds)
                except Exception:
                    continue

            rects = list(self.rectangles)
            def _update():
                self.rect_listbox.delete(0, "end")
                for i, (x0, y0, x1, y1) in enumerate(rects):
                    label = f"[{i+1}]  {x1-x0:.0f} × {y1-y0:.0f}  @ ({x0:.0f}, {y0:.0f})"
                    self.rect_listbox.insert("end", label)
                self._log(f"↺ Scanned: {len(rects)} rectangle(s) found")
            self.after(0, _update)
        except Exception as e:
            err = str(e)
            self.after(0, lambda: self._log(f"✗ Scan error: {err}"))

    # ── PLACE ────────────────────────────────

    def _run(self):
        if not hasattr(self, '_job_queue') or not hasattr(self, '_doc'):
            messagebox.showerror("Error", "Not connected to AutoCAD. Click Reconnect.")
            return

        sel = list(self.rect_listbox.curselection())
        if not sel:
            messagebox.showwarning("No selection", "Select at least one rectangle.")
            return

        fixture_label = self.fixture_var.get()
        block_name    = BLOCK_NAMES.get(fixture_label)
        if not block_name:
            messagebox.showerror("Error", "Invalid fixture type."); return

        try:
            n_lights = int(self.count_var.get())
            assert n_lights > 0
        except Exception:
            messagebox.showerror("Error", "Light count must be a positive integer."); return

        draw_wire   = self.wire_var.get()
        bulge       = self.bulge_var.get()
        draw_label  = self.label_var.get()
        prefix      = self.prefix_var.get().strip()
        label_mode  = self.label_mode_var.get()   # "auto" or "fixed"
        reset_per   = self.reset_var.get()

        try:
            label_start = int(self.start_var.get())
        except Exception:
            label_start = 1

        try:
            text_h  = float(self.texth_var.get())
            text_off= float(self.offset_var.get())
        except Exception:
            text_h, text_off = 250.0, 700.0

        # Post placement job to the COM thread
        try:
            self._job_queue.put(("place", sel, block_name, n_lights, draw_wire, bulge,
                                 draw_label, prefix, label_start, reset_per, text_h, text_off, label_mode))
        except Exception as e:
            messagebox.showerror("Error", f"Could not queue job: {e}")

    def _do_place(self, sel, block_name, n_lights, draw_wire, bulge,
                  draw_label, prefix, label_start, reset_per,
                  text_h, text_off, label_mode="auto"):
        # Already on the COM thread — no CoInitialize needed
        doc = self._doc
        ms  = win32com.client.Dispatch(self._doc.ModelSpace)
        self.after(0, lambda: self._log("─" * 48))

        # Verify block exists
        try:
            self._doc.Blocks.Item(block_name)
        except Exception:
            self.after(0, lambda: self._log(
                f"✗ Block '{block_name}' not found. Insert it manually first."))
            return

        light_layer = LAYER_MAP.get(block_name, "ECLAIRAGE")
        ensure_layer(self._doc, light_layer)
        ensure_layer(self._doc, WIRE_LAYER,  color=6)
        ensure_layer(self._doc, LABEL_LAYER, color=7)

        total       = 0
        all_refs    = []
        label_index = label_start

        for idx in sel:
            x0, y0, x1, y1 = self.rectangles[idx]
            pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

            if reset_per:
                label_index = label_start

            # Blocks
            self._doc.ActiveLayer = self._doc.Layers.Item(light_layer)
            for (px, py) in pts:
                ref = ms.InsertBlock(make_point(px, py, 0.0), block_name,
                                     BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
                all_refs.append(ref)
                total += 1

            # Wire
            if draw_wire and len(pts) >= 2:
                self._doc.ActiveLayer = self._doc.Layers.Item(WIRE_LAYER)
                try:
                    wire = draw_bulge_wire(ms, pts, bulge)
                    if wire:
                        all_refs.append(wire)
                except Exception as we:
                    werr = str(we)
                    self.after(0, lambda: self._log(f"    ✗ Wire error: {werr}"))

            # Labels
            if draw_label:
                self._doc.ActiveLayer = self._doc.Layers.Item(LABEL_LAYER)
                for (px, py) in pts:
                    if label_mode == "fixed":
                        txt = prefix          # exact fixed label, no number
                    else:
                        txt = f"{prefix}.{label_index}"
                        label_index += 1
                    lbl = add_label(ms, px, py, txt, text_h, text_off)
                    if lbl:
                        all_refs.append(lbl)

            msg = (f"✓ Room [{idx+1}]: {len(pts)} lights  {cols}×{rows} grid"
                   f"  wire={'arc' if draw_wire else 'no'}"
                   f"  labels={'yes' if draw_label else 'no'}")
            self.after(0, lambda m=msg: self._log(m))

        self._doc.ActiveLayer = self._doc.Layers.Item("0")
        self._acad.ZoomExtents()
        self._doc.Regen(True)
        self._doc.Save()

        summary = f"✓ Done — {total} light(s) placed and saved."
        self.after(0, lambda: self._log(summary))
        self.after(0, lambda: self._set_status(summary, ok=True))
        self.after(0, lambda: messagebox.showinfo("Done", summary))


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    app = LightingApp()
    app.mainloop()

In [1]:
"""
lighting_app.py
---------------
Tkinter mini-app for placing lights in AutoCAD.
Run while AutoCAD is open with your drawing loaded.

Requirements:
    pip install pyautocad pywin32
    (tkinter is built into Python)
"""

import math
import sys
import array as array_mod
import threading
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import win32com.client
import pythoncom

def make_point(x, y, z=0.0):
    """Create a VARIANT 3D point compatible with raw AutoCAD COM."""
    pt = win32com.client.VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_R8, [x, y, z])
    return pt

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

BLOCK_NAMES = {
    "Panneau LED 60×60":    "PANNEAU_LED_60x60",
    "Spot CoreLine DN140B": "SPOT_CORELINE_DN140B",
    "Hublot étanche 11W":   "HUBLOT_ETANCHE_11W",
    "Applique étanche 11W": "APPLIQUE_ETANCHE_11W",
    "Brasseur d'air 75W":   "BRASSEUR_AIR_75W",
}

LAYER_MAP = {
    "PANNEAU_LED_60x60":    "ECLAIRAGE-PANNEAU",
    "SPOT_CORELINE_DN140B": "ECLAIRAGE-SPOT",
    "HUBLOT_ETANCHE_11W":   "ECLAIRAGE-HUBLOT",
    "APPLIQUE_ETANCHE_11W": "ECLAIRAGE-APPLIQUE",
    "BRASSEUR_AIR_75W":     "ECLAIRAGE-VENTILATEUR",
}

WIRE_LAYER  = "ECLAIRAGE-WIRE"
LABEL_LAYER = "ECLAIRAGE-LABEL"
MARGIN_RATIO = 0.001
BLOCK_SCALE  = 1.0

# ─────────────────────────────────────────────
# CORE LOGIC (same as CLI version)
# ─────────────────────────────────────────────

def ensure_layer(doc, name, color=None):
    try:
        layer = doc.Layers.Item(name)
    except Exception:
        layer = doc.Layers.Add(name)
    if color is not None:
        layer.Color = color
    return layer


def get_rectangle_bounds(entity):
    try:
        if entity.EntityName not in ("AcDbPolyline", "AcDb2dPolyline"):
            return None
        if not entity.Closed:
            return None
        coords = list(entity.Coordinates)
        step = 2 if entity.EntityName == "AcDbPolyline" else 3
        pts  = [(coords[i], coords[i+1]) for i in range(0, len(coords), step)]
        if len(pts) != 4:
            return None
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x0, y0 = min(xs), min(ys)
        x1, y1 = max(xs), max(ys)
        if abs(x1 - x0) < 1e-3 or abs(y1 - y0) < 1e-3:
            return None
        return x0, y0, x1, y1
    except Exception:
        return None


def grid_points(x_min, y_min, x_max, y_max, n):
    width  = x_max - x_min
    height = y_max - y_min
    mx = width  * MARGIN_RATIO
    my = height * MARGIN_RATIO
    ux0, uy0 = x_min + mx, y_min + my
    ux1, uy1 = x_max - mx, y_max - my
    uw = ux1 - ux0
    uh = uy1 - uy0

    best_cols, best_rows, best_score = 1, n, float("inf")
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        waste = cols * rows - n
        aspect_diff = abs((cols / rows) - (uw / uh)) if uh > 0 else cols
        score = aspect_diff + waste * 0.5
        if score < best_score:
            best_score = score
            best_cols, best_rows = cols, rows

    cols, rows = best_cols, best_rows
    cell_w = uw / cols
    cell_h = uh / rows

    pts = []
    for r in range(rows):
        row_pts = []
        for c in range(cols):
            if len(pts) + len(row_pts) >= n:
                break
            cx = ux0 + cell_w * c + cell_w / 2
            cy = uy0 + cell_h * r + cell_h / 2
            row_pts.append((cx, cy))
        if r % 2 == 1:
            row_pts.reverse()
        pts.extend(row_pts)
        if len(pts) >= n:
            break

    return pts[:n], cols, rows


def draw_bulge_wire(ms, pts, bulge):
    if len(pts) < 2:
        return None
    flat_pts = []
    for (x, y) in pts:
        flat_pts.extend([x, y])
    bulge_vals = []
    for i in range(len(pts) - 1):
        sign = 1 if i % 2 == 0 else -1
        bulge_vals.append(sign * abs(bulge))
    bulge_vals.append(0.0)

    # AutoCAD COM requires a VARIANT array of doubles
    pt_variant = win32com.client.VARIANT(
        pythoncom.VT_ARRAY | pythoncom.VT_R8, flat_pts)

    pline = ms.AddLightWeightPolyline(pt_variant)
    pline = win32com.client.Dispatch(pline)
    pline.Closed = False
    for i, b in enumerate(bulge_vals):
        pline.SetBulge(i, b)
    pline.Update()
    return pline


def add_label(ms, x, y, text, height, offset_y, offset_x=0.0):
    try:
        lbl = ms.AddMText(make_point(x + offset_x, y - offset_y, 0.0), height * 8, text)
        lbl.Height          = height
        lbl.AttachmentPoint = 5
        lbl.Color           = 7
        # Apply Times New Roman Bold using MText formatting codes
        # {\fTimes New Roman|b1|i0|c0|p0; ...text... } sets font+bold for the content
        lbl.TextString = "{\\fTimes New Roman|b1|i0;" + text + "}"
        return lbl
    except Exception:
        return None


# ─────────────────────────────────────────────
# TKINTER APP
# ─────────────────────────────────────────────

class LightingApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("AutoCAD Lighting Placer")
        self.resizable(False, False)
        self.configure(bg="#1e1e2e")

        self.rectangles = []   # list of bounds tuples (plain Python, thread-safe)
        self._acad = None
        self._doc  = None
        self._ms   = None
        import queue
        self._job_queue = queue.Queue()

        self._build_ui()
        self._connect_autocad()

    # ── UI BUILD ────────────────────────────

    def _build_ui(self):
        PAD  = 12
        FONT = ("Segoe UI", 10)
        FONT_BOLD = ("Segoe UI", 10, "bold")
        BG   = "#1e1e2e"
        CARD = "#2a2a3e"
        ACC  = "#7c3aed"   # purple accent
        FG   = "#e2e8f0"
        ENTRY_BG = "#313145"

        style = ttk.Style(self)
        style.theme_use("clam")
        style.configure("TLabel",      background=CARD,  foreground=FG,  font=FONT)
        style.configure("TFrame",      background=CARD)
        style.configure("TLabelframe", background=CARD,  foreground=FG,  font=FONT_BOLD)
        style.configure("TLabelframe.Label", background=CARD, foreground=ACC, font=FONT_BOLD)
        style.configure("TCombobox",   fieldbackground=ENTRY_BG, background=ENTRY_BG,
                        foreground=FG, font=FONT)
        style.configure("TCheckbutton", background=CARD, foreground=FG, font=FONT)
        style.map("TCheckbutton", background=[("active", CARD)])

        outer = tk.Frame(self, bg=BG, padx=PAD, pady=PAD)
        outer.pack(fill="both", expand=True)

        # ── Header ──────────────────────────
        hdr = tk.Frame(outer, bg=ACC, pady=8)
        hdr.pack(fill="x", pady=(0, PAD))
        tk.Label(hdr, text="⚡  AutoCAD Lighting Placer",
                 font=("Segoe UI", 14, "bold"), bg=ACC, fg="white").pack()

        # ── Status bar ──────────────────────
        self.status_var = tk.StringVar(value="Connecting to AutoCAD…")
        status_bar = tk.Frame(outer, bg=CARD, padx=8, pady=5)
        status_bar.pack(fill="x", pady=(0, PAD))
        self.status_dot = tk.Label(status_bar, text="●", font=("Segoe UI", 12),
                                   bg=CARD, fg="#f59e0b")
        self.status_dot.pack(side="left")
        tk.Label(status_bar, textvariable=self.status_var,
                 font=FONT, bg=CARD, fg=FG).pack(side="left", padx=6)

        # ── Two columns ─────────────────────
        cols = tk.Frame(outer, bg=BG)
        cols.pack(fill="both")

        left  = tk.Frame(cols, bg=BG)
        right = tk.Frame(cols, bg=BG)
        left.pack(side="left", fill="both", padx=(0, 6))
        right.pack(side="left", fill="both")

        # ── LEFT: Room + Fixture ─────────────
        room_frame = ttk.LabelFrame(left, text="  Room Selection", padding=10)
        room_frame.pack(fill="x", pady=(0, 8))

        tk.Label(room_frame, text="Rectangles found:", bg=CARD, fg=FG,
                 font=FONT).grid(row=0, column=0, sticky="w", pady=2)

        self.rect_listbox = tk.Listbox(
            room_frame, height=5, selectmode="multiple",
            bg=ENTRY_BG, fg=FG, font=FONT,
            selectbackground=ACC, selectforeground="white",
            borderwidth=0, highlightthickness=1,
            highlightcolor=ACC, relief="flat"
        )
        self.rect_listbox.grid(row=1, column=0, columnspan=2, sticky="ew", pady=4)

        btn_row = tk.Frame(room_frame, bg=CARD)
        btn_row.grid(row=2, column=0, columnspan=2, sticky="ew")
        self._btn(btn_row, "⟳  Scan", self._scan_rectangles, ACC).pack(side="left", padx=(0,4))
        self._btn(btn_row, "Select All", self._select_all).pack(side="left", padx=(4,0))
        self._btn(btn_row, "⚡ Reconnect", self._connect_autocad, "#b45309").pack(side="left", padx=(4,0))

        fix_frame = ttk.LabelFrame(left, text="  Fixture", padding=10)
        fix_frame.pack(fill="x", pady=(0, 8))

        tk.Label(fix_frame, text="Type:", bg=CARD, fg=FG, font=FONT).grid(
            row=0, column=0, sticky="w", pady=2)
        self.fixture_var = tk.StringVar()
        fix_combo = ttk.Combobox(fix_frame, textvariable=self.fixture_var,
                                 values=list(BLOCK_NAMES.keys()),
                                 state="readonly", width=26)
        fix_combo.grid(row=0, column=1, sticky="ew", padx=(6,0))
        fix_combo.current(0)

        tk.Label(fix_frame, text="Count per room:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.count_var = tk.StringVar(value="6")
        self._entry(fix_frame, self.count_var).grid(row=1, column=1, sticky="ew", padx=(6,0))

        # ── RIGHT: Wire + Label ──────────────
        wire_frame = ttk.LabelFrame(right, text="  Wire", padding=10)
        wire_frame.pack(fill="x", pady=(0, 8))

        self.wire_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(wire_frame, text="Draw arc wire", variable=self.wire_var,
                        command=self._toggle_wire).grid(row=0, column=0, columnspan=2,
                                                        sticky="w", pady=2)

        tk.Label(wire_frame, text="Bulge:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.bulge_var = tk.DoubleVar(value=0.5)
        self.bulge_scale = tk.Scale(
            wire_frame, from_=-1.0, to=1.0, resolution=0.05,
            orient="horizontal", variable=self.bulge_var,
            bg=CARD, fg=FG, troughcolor=ENTRY_BG,
            highlightthickness=0, activebackground=ACC,
            length=160, font=("Segoe UI", 8)
        )
        self.bulge_scale.grid(row=1, column=1, sticky="ew", padx=(6,0))

        self.bulge_lbl = tk.Label(wire_frame, text="≈ 106° arc",
                                  bg=CARD, fg="#94a3b8", font=("Segoe UI", 9))
        self.bulge_lbl.grid(row=2, column=1, sticky="w", padx=(6,0))
        self.bulge_var.trace_add("write", self._update_bulge_label)

        lbl_frame = ttk.LabelFrame(right, text="  Circuit Labels", padding=10)
        lbl_frame.pack(fill="x", pady=(0, 8))

        self.label_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(lbl_frame, text="Add labels", variable=self.label_var,
                        command=self._toggle_labels).grid(row=0, column=0, columnspan=2,
                                                          sticky="w", pady=2)

        # Label mode: Fixed or Auto-increment
        tk.Label(lbl_frame, text="Mode:", bg=CARD, fg=FG, font=FONT).grid(
            row=1, column=0, sticky="w", pady=2)
        self.label_mode_var = tk.StringVar(value="auto")
        mode_frame = tk.Frame(lbl_frame, bg=CARD)
        mode_frame.grid(row=1, column=1, sticky="w", padx=(6,0))
        ttk.Radiobutton(mode_frame, text="Auto", variable=self.label_mode_var,
                        value="auto", command=self._toggle_label_mode).pack(side="left")
        ttk.Radiobutton(mode_frame, text="Fixed", variable=self.label_mode_var,
                        value="fixed", command=self._toggle_label_mode).pack(side="left", padx=(8,0))

        tk.Label(lbl_frame, text="Prefix:", bg=CARD, fg=FG, font=FONT).grid(
            row=2, column=0, sticky="w", pady=2)
        self.prefix_var = tk.StringVar(value="E20")
        self._entry(lbl_frame, self.prefix_var, width=8).grid(row=2, column=1,
                                                               sticky="w", padx=(6,0))

        # Auto-increment fields
        self.start_lbl = tk.Label(lbl_frame, text="Start #:", bg=CARD, fg=FG, font=FONT)
        self.start_lbl.grid(row=3, column=0, sticky="w", pady=2)
        self.start_var = tk.StringVar(value="1")
        self.start_entry = self._entry(lbl_frame, self.start_var, width=5)
        self.start_entry.grid(row=3, column=1, sticky="w", padx=(6,0))

        self.reset_var = tk.BooleanVar(value=False)
        self.reset_chk = ttk.Checkbutton(lbl_frame, text="Reset counter per room",
                                          variable=self.reset_var)
        self.reset_chk.grid(row=4, column=0, columnspan=2, sticky="w", pady=2)

        tk.Label(lbl_frame, text="Text height:", bg=CARD, fg=FG, font=FONT).grid(
            row=5, column=0, sticky="w", pady=2)
        self.texth_var = tk.StringVar(value="250")
        self._entry(lbl_frame, self.texth_var, width=8).grid(row=5, column=1,
                                                              sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="Y Offset:", bg=CARD, fg=FG, font=FONT).grid(
            row=6, column=0, sticky="w", pady=2)
        self.offset_var = tk.StringVar(value="700")
        self._entry(lbl_frame, self.offset_var, width=8).grid(row=6, column=1,
                                                               sticky="w", padx=(6,0))

        tk.Label(lbl_frame, text="X Offset:", bg=CARD, fg=FG, font=FONT).grid(
            row=7, column=0, sticky="w", pady=2)
        self.xoffset_var = tk.StringVar(value="0")
        self._entry(lbl_frame, self.xoffset_var, width=8).grid(row=7, column=1,
                                                                sticky="w", padx=(6,0))

        # ── Place button ────────────────────
        btn_frame = tk.Frame(outer, bg=BG, pady=8)
        btn_frame.pack(fill="x")
        self._btn(btn_frame, "⚡  Place Lights", self._run, ACC,
                  font=("Segoe UI", 11, "bold"), pady=10).pack(fill="x")

        # ── Log ─────────────────────────────
        log_frame = ttk.LabelFrame(outer, text="  Log", padding=6)
        log_frame.pack(fill="both", expand=True, pady=(8,0))
        self.log = scrolledtext.ScrolledText(
            log_frame, height=8, bg="#0f0f1a", fg="#a5f3fc",
            font=("Consolas", 9), borderwidth=0, relief="flat",
            insertbackground="white"
        )
        self.log.pack(fill="both", expand=True)
        self.log.configure(state="disabled")

    def _btn(self, parent, text, cmd, bg="#3f3f5a", font=("Segoe UI", 10), pady=6):
        return tk.Button(parent, text=text, command=cmd,
                         bg=bg, fg="white", font=font,
                         relief="flat", cursor="hand2",
                         activebackground="#5b5b7a",
                         activeforeground="white",
                         padx=12, pady=pady, bd=0)

    def _entry(self, parent, var, width=12):
        return tk.Entry(parent, textvariable=var, width=width,
                        bg="#313145", fg="#e2e8f0", font=("Segoe UI", 10),
                        relief="flat", insertbackground="white",
                        highlightthickness=1, highlightcolor="#7c3aed")

    # ── HELPERS ─────────────────────────────

    def _log(self, msg, color=None):
        self.log.configure(state="normal")
        self.log.insert("end", msg + "\n")
        self.log.see("end")
        self.log.configure(state="disabled")

    def _set_status(self, msg, ok=True):
        self.status_var.set(msg)
        self.status_dot.configure(fg="#22c55e" if ok else "#ef4444")

    def _update_bulge_label(self, *_):
        b = self.bulge_var.get()
        if abs(b) < 0.01:
            txt = "straight line"
        else:
            deg = math.degrees(4 * math.atan(abs(b)))
            txt = f"≈ {deg:.0f}° arc"
        self.bulge_lbl.configure(text=txt)

    def _toggle_wire(self):
        state = "normal" if self.wire_var.get() else "disabled"
        self.bulge_scale.configure(state=state)

    def _toggle_labels(self):
        pass  # fields stay visible; logic skips if unchecked

    def _toggle_label_mode(self):
        """Show/hide auto-increment fields based on selected mode."""
        is_auto = self.label_mode_var.get() == "auto"
        state = "normal" if is_auto else "disabled"
        self.start_entry.configure(state=state)
        self.reset_chk.configure(state=state)
        self.start_lbl.configure(fg="#e2e8f0" if is_auto else "#555570")

    def _select_all(self):
        self.rect_listbox.select_set(0, "end")

    # ── AUTOCAD CONNECTION ───────────────────

    def _connect_autocad(self):
        """Launch one background thread that owns ALL COM work: connect + scan."""
        threading.Thread(target=self._com_worker, daemon=True).start()

    def _com_worker(self):
        """
        Single thread that owns every COM call.
        Rule: COM objects are NEVER passed to other threads.
        All AutoCAD work (connect, scan, place) happens here.
        """
        pythoncom.CoInitialize()
        try:
            acad = win32com.client.Dispatch(
                win32com.client.GetActiveObject("AutoCAD.Application"))
            doc  = win32com.client.Dispatch(acad.ActiveDocument)
            ms   = win32com.client.Dispatch(doc.ModelSpace)
        except Exception as ex:
            msg = str(ex)
            self.after(0, lambda: self._set_status("Not connected — click Reconnect", ok=False))
            self.after(0, lambda: self._log(f"✗ Cannot connect: {msg}"))
            return

        # Store references — only used from this thread via the queue
        self._acad = acad
        self._doc  = doc
        self._ms   = ms

        name = doc.Name
        self.after(0, lambda: self._set_status(f"Connected: {name}", ok=True))
        self.after(0, lambda: self._log(f"✓ Connected to {name}"))

        # Immediately scan
        self._do_scan()

        # Event loop — wait for jobs from the UI thread
        while True:
            try:
                job = self._job_queue.get(timeout=0.2)
                if job is None:
                    break
                if job[0] == "scan":
                    self._do_scan()
                elif job[0] == "place":
                    self._do_place(*job[1:])
            except Exception:
                continue

    # ── SCAN ────────────────────────────────

    def _scan_rectangles(self):
        """Called from UI — posts a scan job to the COM thread via queue."""
        try:
            self._job_queue.put(("scan",))
        except Exception:
            self._log("✗ Not connected yet — click Reconnect")

    def _do_scan(self):
        """Runs on the COM thread."""
        self.rectangles = []
        try:
            ms    = win32com.client.Dispatch(self._doc.ModelSpace)
            count = ms.Count
            for i in range(count):
                try:
                    entity = win32com.client.Dispatch(ms.Item(i))
                    bounds = get_rectangle_bounds(entity)
                    if bounds:
                        # Store only the bounds (plain Python data, safe to share)
                        self.rectangles.append(bounds)
                except Exception:
                    continue

            rects = list(self.rectangles)
            def _update():
                self.rect_listbox.delete(0, "end")
                for i, (x0, y0, x1, y1) in enumerate(rects):
                    label = f"[{i+1}]  {x1-x0:.0f} × {y1-y0:.0f}  @ ({x0:.0f}, {y0:.0f})"
                    self.rect_listbox.insert("end", label)
                self._log(f"↺ Scanned: {len(rects)} rectangle(s) found")
            self.after(0, _update)
        except Exception as e:
            err = str(e)
            self.after(0, lambda: self._log(f"✗ Scan error: {err}"))

    # ── PLACE ────────────────────────────────

    def _run(self):
        if not hasattr(self, '_job_queue') or not hasattr(self, '_doc'):
            messagebox.showerror("Error", "Not connected to AutoCAD. Click Reconnect.")
            return

        sel = list(self.rect_listbox.curselection())
        if not sel:
            messagebox.showwarning("No selection", "Select at least one rectangle.")
            return

        fixture_label = self.fixture_var.get()
        block_name    = BLOCK_NAMES.get(fixture_label)
        if not block_name:
            messagebox.showerror("Error", "Invalid fixture type."); return

        try:
            n_lights = int(self.count_var.get())
            assert n_lights > 0
        except Exception:
            messagebox.showerror("Error", "Light count must be a positive integer."); return

        draw_wire   = self.wire_var.get()
        bulge       = self.bulge_var.get()
        draw_label  = self.label_var.get()
        prefix      = self.prefix_var.get().strip()
        label_mode  = self.label_mode_var.get()   # "auto" or "fixed"
        reset_per   = self.reset_var.get()

        try:
            label_start = int(self.start_var.get())
        except Exception:
            label_start = 1

        try:
            text_h   = float(self.texth_var.get())
            text_off = float(self.offset_var.get())
            text_xoff= float(self.xoffset_var.get())
        except Exception:
            text_h, text_off, text_xoff = 250.0, 700.0, 0.0

        # Post placement job to the COM thread
        try:
            self._job_queue.put(("place", sel, block_name, n_lights, draw_wire, bulge,
                                 draw_label, prefix, label_start, reset_per, text_h, text_off, text_xoff, label_mode))
        except Exception as e:
            messagebox.showerror("Error", f"Could not queue job: {e}")

    def _do_place(self, sel, block_name, n_lights, draw_wire, bulge,
                  draw_label, prefix, label_start, reset_per,
                  text_h, text_off, text_xoff=0.0, label_mode="auto"):
        # Already on the COM thread — no CoInitialize needed
        doc = self._doc
        ms  = win32com.client.Dispatch(self._doc.ModelSpace)
        self.after(0, lambda: self._log("─" * 48))

        # Verify block exists
        try:
            self._doc.Blocks.Item(block_name)
        except Exception:
            self.after(0, lambda: self._log(
                f"✗ Block '{block_name}' not found. Insert it manually first."))
            return

        light_layer = LAYER_MAP.get(block_name, "ECLAIRAGE")
        ensure_layer(self._doc, light_layer)
        ensure_layer(self._doc, WIRE_LAYER,  color=6)
        ensure_layer(self._doc, LABEL_LAYER, color=7)

        total       = 0
        all_refs    = []
        label_index = label_start

        for idx in sel:
            x0, y0, x1, y1 = self.rectangles[idx]
            pts, cols, rows = grid_points(x0, y0, x1, y1, n_lights)

            if reset_per:
                label_index = label_start

            # Blocks
            self._doc.ActiveLayer = self._doc.Layers.Item(light_layer)
            for (px, py) in pts:
                ref = ms.InsertBlock(make_point(px, py, 0.0), block_name,
                                     BLOCK_SCALE, BLOCK_SCALE, BLOCK_SCALE, 0.0)
                all_refs.append(ref)
                total += 1

            # Wire
            if draw_wire and len(pts) >= 2:
                self._doc.ActiveLayer = self._doc.Layers.Item(WIRE_LAYER)
                try:
                    wire = draw_bulge_wire(ms, pts, bulge)
                    if wire:
                        all_refs.append(wire)
                except Exception as we:
                    werr = str(we)
                    self.after(0, lambda: self._log(f"    ✗ Wire error: {werr}"))

            # Labels
            if draw_label:
                self._doc.ActiveLayer = self._doc.Layers.Item(LABEL_LAYER)
                for (px, py) in pts:
                    if label_mode == "fixed":
                        txt = prefix          # exact fixed label, no number
                    else:
                        txt = f"{prefix}.{label_index}"
                        label_index += 1
                    lbl = add_label(ms, px, py, txt, text_h, text_off, text_xoff)
                    if lbl:
                        all_refs.append(lbl)

            msg = (f"✓ Room [{idx+1}]: {len(pts)} lights  {cols}×{rows} grid"
                   f"  wire={'arc' if draw_wire else 'no'}"
                   f"  labels={'yes' if draw_label else 'no'}")
            self.after(0, lambda m=msg: self._log(m))

        self._doc.ActiveLayer = self._doc.Layers.Item("0")
        self._acad.ZoomExtents()
        self._doc.Regen(True)
        self._doc.Save()

        summary = f"✓ Done — {total} light(s) placed and saved."
        self.after(0, lambda: self._log(summary))
        self.after(0, lambda: self._set_status(summary, ok=True))
        self.after(0, lambda: messagebox.showinfo("Done", summary))


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    app = LightingApp()
    app.mainloop()